> **Runs automatically.** This copy lives in the `football-picks` repository on GitHub, which runs it on a schedule and keeps the logs in the repository's `data` folder. Open it here to read or change it; the schedule picks up whatever is saved in the repository.

# NFL game model v7: winners, spreads, totals and player props

This notebook replaces `nfl_game_model_v6`. **The models are the same as version 6**, and the picks log under the same names. What's new:
- **Website data.** Step 12 saves this week's games, the player props, the season record and every graded game to `app_data/nfl.json` in your Google Drive. The picks website reads this file.
- **Projected scores show one decimal**, like "HOU 19.5, JAX 23.5", so they always agree with the spread.
- **Sportsbook prop lines.** With an `ODDS_API_KEY` secret, Step 10 fetches each prop's line and prices from The Odds API, so the props table and the props log compare the model with real lines.
- **Top picks inputs.** Step 12 also saves each game's chance to cover the spread and each player's chance of reaching common milestones, for the website's Top picks lists.

Version 6 added the plain-language game cards in Step 7. It also tested points ratings and success-rate ratings in the margin model; they didn't hold up on 2020–2025, so the margin model is unchanged (Step 4 has the numbers).

**What it does**
1. Loads every NFL game since 1999, plus every player's game stats, from nflverse.
2. Rates every team with Elo, using the settings from the baseline notebook.
3. Rates every quarterback from his recent games.
4. Combines Elo, quarterbacks, rest and home field into a model of the final margin.
5. Tests the margin model on 2020–2025 against Elo and the market.
6. Builds and tests the totals model the same way.
7. Shows this week's games with a projected score for each.
8. Saves every model's picks to your pick log and grades past picks.
9. Builds and tests the player props model.
10. Shows this week's player props.
11. Saves the props to a props log and grades past props.
12. Saves everything the website needs to one file.

**To run it:** in Colab, choose **Runtime → Run all**. It takes a few minutes. Colab asks for permission to use your Google Drive; allow it so the logs can be saved.

**Every week:** run it before the week's first game (usually Thursday night), then again Sunday morning so the props pick up the final injury reports. Versions 5 and 6 log exactly the same picks, so running one by mistake does no harm, but only this version updates the website file. After the games, run `game_review_v1` to see what the models got right and wrong. Run it before this notebook, and its explanations show up on the website too. If a starting quarterback is ruled out after the line comes out, see `QB_OVERRIDES` in Step 3. For other players, see `PLAYERS_OUT` in Step 10.

In [ ]:
%pip install -q nflreadpy "polars>=1.0"

## Step 1: Load the games and player stats

`games` has one row per game, with the score, betting lines, rest days and each team's starting quarterback. For past games that's who actually started. For next week, nflverse lists the projected starter.

`player_stats` has one row per player per game. The quarterback ratings in Step 3 come from it.

In [ ]:
import math
import itertools
import numpy as np
import polars as pl
import nflreadpy as nfl
from scipy.stats import norm

pl.Config.set_tbl_rows(40)
pl.Config.set_tbl_cols(20)

# Teams that moved keep one continuous rating history
RELOCATED = {"OAK": "LV", "SD": "LAC", "STL": "LA"}

games = (
    nfl.load_schedules()
    .with_columns(
        pl.col("home_team").replace(RELOCATED),
        pl.col("away_team").replace(RELOCATED),
    )
    .sort(["season", "gameday", "gametime", "game_id"])
)
player_stats = nfl.load_player_stats(True)

print(f"{games.height:,} games and {player_stats.height:,} player stat lines, seasons {games['season'].min()}–{games['season'].max()}")

## Step 2: Elo ratings

This is the same Elo model as the baseline notebook. Every team starts at 1500, the winner takes points from the loser after each game, and ratings drift partway back to 1500 each offseason. The settings were tuned on 2006–2019 in the baseline notebook, so they're fixed here.

Each game is predicted *before* its result updates the ratings, so the model never sees the future.

In [ ]:
ELO_SETTINGS = {"k": 20, "home_field": 48, "regress": 0.5}   # tuned on 2006–2019 in the baseline notebook


def run_elo(games, k=20, home_field=48, regress=0.5, mean=1500):
    """Predict every game in date order, then update ratings once it's played."""
    ratings = {}
    season = None
    rows = []
    for g in games.iter_rows(named=True):
        if g["season"] != season:                       # new season: pull every team toward 1500
            season = g["season"]
            ratings = {t: mean + (1 - regress) * (r - mean) for t, r in ratings.items()}

        home, away = g["home_team"], g["away_team"]
        r_home, r_away = ratings.get(home, mean), ratings.get(away, mean)
        edge = home_field if g["location"] == "Home" else 0
        diff = r_home + edge - r_away
        p_home = 1 / (1 + 10 ** (-diff / 400))           # the prediction, made BEFORE the game
        rows.append({"game_id": g["game_id"], "elo_home": r_home, "elo_away": r_away,
                     "elo_p_home": p_home, "elo_spread": diff / 25})

        if g["result"] is None:
            continue
        margin = g["result"]
        actual = 1.0 if margin > 0 else 0.0 if margin < 0 else 0.5
        winner_diff = diff if margin > 0 else -diff
        mov_mult = math.log(abs(margin) + 1) * 2.2 / (winner_diff * 0.001 + 2.2)
        shift = k * mov_mult * (actual - p_home)
        ratings[home] = r_home + shift
        ratings[away] = r_away - shift
    return pl.DataFrame(rows), ratings


elo_preds, elo_ratings = run_elo(games, **ELO_SETTINGS)
df = games.join(elo_preds, on="game_id")

## Step 3: Quarterback ratings

Elo rates the whole team, so when a starting quarterback gets hurt it takes Elo several losses to notice. This step rates every quarterback on his own.

**How a rating works**
- It measures **EPA per play**: expected points added on his passes, sacks and runs. A rating of +0.10 means his plays add about a tenth of a point more than the league average.
- It's compared with **last season's** league average, so nothing from the future leaks in.
- Recent games count more. Each older game fades by 2%.
- Every quarterback starts with 50 plays at −0.10, about backup level, so a newcomer is rated like a backup until he proves otherwise.
- Only games **before** the one being predicted are used.

**Who starts:** nflverse lists each game's starter. For past games that's who actually started; for next week it's a projection. If a team's starter changes after that, for example a late injury, name the new starter in `QB_OVERRIDES`, spelled the way nflverse writes it, and clear it once the week is over.

In [ ]:
QB_SETTINGS = {"decay": 0.98, "prior_plays": 50, "prior": -0.10}   # chosen on 2016–2019 practice seasons (Step 4)

# Late news only. Example: QB_OVERRIDES = {"SEA": "Sam Darnold"}
QB_OVERRIDES = {}

# One row per player per game with at least one pass attempt or sack
qb_games = (
    player_stats.with_columns(
        dropbacks=pl.col("attempts").fill_null(0) + pl.col("sacks_suffered").fill_null(0),
        plays=pl.col("attempts").fill_null(0) + pl.col("sacks_suffered").fill_null(0) + pl.col("carries").fill_null(0),
        epa=pl.col("passing_epa").fill_null(0) + pl.col("rushing_epa").fill_null(0),
    )
    .filter(pl.col("dropbacks") > 0)
    .select("player_id", "player_display_name", "game_id", "season", "plays", "epa")
)

# League average EPA per play, using last season's number
league = (
    qb_games.group_by("season")
    .agg(avg=pl.col("epa").sum() / pl.col("plays").sum())
    .sort("season")
    .with_columns(prev_avg=pl.col("avg").shift(1))
    .with_columns(prev_avg=pl.col("prev_avg").fill_null(pl.col("avg")))
)
qb_games = qb_games.join(league.select("season", "prev_avg"), on="season").with_columns(
    epa_vs_avg=pl.col("epa") - pl.col("plays") * pl.col("prev_avg")
)

qb_plays_by_game = {}
for r in qb_games.iter_rows(named=True):
    qb_plays_by_game.setdefault(r["game_id"], []).append((r["player_id"], r["plays"], r["epa_vs_avg"]))

qb_names = dict(qb_games.select("player_id", "player_display_name").iter_rows())
qb_ids = {name: pid for pid, name in qb_games.sort("season").select("player_id", "player_display_name").iter_rows()}


def qb_ratings(games, decay=0.98, prior_plays=50, prior=-0.10, overrides=None):
    """Each starter's rating BEFORE the game, plus how it compares with the team's previous starter."""
    overrides = overrides or {}
    sum_epa, sum_plays, last_starter, rows = {}, {}, {}, []

    def rating(qb):
        return (sum_epa.get(qb, 0) + prior * prior_plays) / (sum_plays.get(qb, 0) + prior_plays)

    for g in games.iter_rows(named=True):
        row = {"game_id": g["game_id"]}
        for side in ("home", "away"):
            team = g[f"{side}_team"]
            qb = g[f"{side}_qb_id"] or last_starter.get(team)
            if g["result"] is None and team in overrides:
                qb = qb_ids.get(overrides[team], "new: " + overrides[team])
            usual = last_starter.get(team, qb)
            row[f"{side}_qb"] = qb
            row[f"{side}_qb_rating"] = rating(qb)
            row[f"{side}_qb_change"] = rating(qb) - rating(usual)   # 0 when last game's starter starts again
        rows.append(row)

        if g["result"] is None:
            continue
        for side in ("home", "away"):
            if g[f"{side}_qb_id"]:
                last_starter[g[f"{side}_team"]] = g[f"{side}_qb_id"]
        for qb, plays, epa in qb_plays_by_game.get(g["game_id"], []):
            sum_epa[qb] = decay * sum_epa.get(qb, 0) + epa
            sum_plays[qb] = decay * sum_plays.get(qb, 0) + plays
    return pl.DataFrame(rows)


for team, name in QB_OVERRIDES.items():
    note = "" if name in qb_ids else "  <- not found in nflverse stats, so he's rated as a new QB. Check the spelling."
    print(f"Override: {team} starts {name}{note}")

qb = qb_ratings(games, **QB_SETTINGS, overrides=QB_OVERRIDES)

## Step 4: The model

The model predicts the **final margin** (home score minus away score) from five inputs, all known before kickoff:

| Input | What it means |
|---|---|
| `elo_pts` | Elo gap between the teams, in points |
| `home` | 1 for a true home game, 0 at a neutral site |
| `rest_diff` | Home team's days of rest minus the away team's, capped at 7 |
| `qb_diff` | Home starter's rating minus the away starter's |
| `qb_change_diff` | How much each team's starter differs from the one who started its last game |

It's a linear regression: each input gets a weight, learned from past games. The predicted margin is the model's spread. The **win probability** comes from that same margin, using how far real margins typically land from predictions (about 13.5 points). That keeps the spread and the win chance consistent with each other.

**How the settings were chosen:** the model was trained on 2006–2015 and checked on 2016–2019, a practice exam that 2020–2025 never touches. On those practice seasons:
- Adding quarterbacks cut log loss from 0.629 to 0.619. Rest days barely moved it; they stay in because the effect is small but sensible, about 0.2 points per extra day of rest.
- Gradient boosting did worse than the linear model (0.621), so the simpler model stays.
- **Version 6 tested points ratings**, the scoring offense and defense ratings the college model uses, as a sixth input. The post-game review had hinted that the model adjusts too slowly. On the practice seasons they helped a little: log loss 0.619 to 0.618, spread miss 10.00 to 9.94 points. Blending in success-rate "deserved points" like the college model's added nothing more. **On 2020–2025 neither held up**: log loss went from 0.628 to 0.630 with points ratings and 0.629 with the success-rate blend, and the spread miss from 10.09 to 10.10. So the margin model stays as it is.

To redo that search yourself, set `RUN_SEARCH = True` in the second cell below. It adds about a minute.

In [ ]:
FEATURES = ["elo_pts", "home", "rest_diff", "qb_diff", "qb_change_diff"]


def add_features(df, qb):
    return df.join(qb, on="game_id").with_columns(
        elo_pts=(pl.col("elo_home") - pl.col("elo_away")) / 25,
        home=(pl.col("location") == "Home").cast(pl.Float64),
        rest_diff=(pl.col("home_rest") - pl.col("away_rest")).clip(-7, 7).cast(pl.Float64),
        qb_diff=pl.col("home_qb_rating") - pl.col("away_qb_rating"),
        qb_change_diff=pl.col("home_qb_change") - pl.col("away_qb_change"),
    )


def fit_margin_model(data, seasons, features=FEATURES):
    """Learn one weight per input from the given seasons. Returns the weights and the typical miss (sigma)."""
    train = data.filter(pl.col("season").is_between(*seasons) & pl.col("result").is_not_null())
    X = train.select(features).to_numpy()
    y = train["result"].to_numpy().astype(float)
    weights, *_ = np.linalg.lstsq(X, y, rcond=None)
    sigma = float(np.std(y - X @ weights))
    return weights, sigma


def predict(data, weights, sigma, features=FEATURES):
    margin = data.select(features).to_numpy() @ weights
    return data.with_columns(
        model_spread=pl.Series(margin),                       # positive = home team favored
        model_p_home=pl.Series(norm.cdf(margin / sigma)),     # chance the home team wins
    )


def log_loss(p, y):
    p = p.clip(1e-12, 1 - 1e-12)
    return float(-(y * p.log() + (1 - y) * (1 - p).log()).mean())


def brier(p, y):
    return float(((p - y) ** 2).mean())


def accuracy(p, y):
    return float(((p > 0.5) == (y == 1)).mean())


def decided(df):
    """Games with a final score that didn't end in a tie."""
    return df.filter(pl.col("result").is_not_null() & (pl.col("result") != 0)).with_columns(
        home_won=(pl.col("result") > 0).cast(pl.Float64)
    )


data = add_features(df, qb)

In [ ]:
RUN_SEARCH = False   # True = redo the practice-season search (about a minute)

if RUN_SEARCH:
    from sklearn.ensemble import HistGradientBoostingRegressor

    PRACTICE_TRAIN, PRACTICE_TEST = (2006, 2015), (2016, 2019)

    def practice_log_loss(d, features):
        w, s = fit_margin_model(d, PRACTICE_TRAIN, features)
        p = decided(predict(d.filter(pl.col("season").is_between(*PRACTICE_TEST)), w, s, features))
        return round(log_loss(p["model_p_home"], p["home_won"]), 4)

    rows = []
    for decay, prior_plays, prior in itertools.product([0.97, 0.98, 0.99], [25, 50, 100], [-0.10, -0.05, 0.0]):
        d = add_features(df, qb_ratings(games, decay, prior_plays, prior))
        rows.append({"decay": decay, "prior_plays": prior_plays, "prior": prior, "log_loss": practice_log_loss(d, FEATURES)})
    print("Quarterback rating settings, best first:")
    display(pl.DataFrame(rows).sort("log_loss").head(5))

    print("Which inputs help:")
    display(pl.DataFrame([
        {"inputs": "Elo + home", "log_loss": practice_log_loss(data, ["elo_pts", "home"])},
        {"inputs": "+ rest", "log_loss": practice_log_loss(data, ["elo_pts", "home", "rest_diff"])},
        {"inputs": "+ rest + quarterbacks", "log_loss": practice_log_loss(data, FEATURES)},
    ]))

    train = data.filter(pl.col("season").is_between(*PRACTICE_TRAIN) & pl.col("result").is_not_null())
    gb = HistGradientBoostingRegressor(max_depth=2, learning_rate=0.05, max_iter=300, min_samples_leaf=40, random_state=0)
    gb.fit(train.select(FEATURES).to_numpy(), train["result"].to_numpy())
    s = float(np.std(train["result"].to_numpy() - gb.predict(train.select(FEATURES).to_numpy())))
    test = decided(data.filter(pl.col("season").is_between(*PRACTICE_TEST)))
    p = pl.Series(norm.cdf(gb.predict(test.select(FEATURES).to_numpy()) / s))
    print("Gradient boosting log loss:", round(log_loss(p, test["home_won"]), 4))

## Step 5: The final test, 2020–2025

The model is trained once on 2006–2019, then scored on 2020–2025, the same games and the same measures as the baseline notebook. Lower log loss and Brier are better; a coin flip scores 0.693 and 0.25.

Betting odds become probabilities the same way as before: convert each side's odds, then remove the sportsbook's cut.

In [ ]:
def american_to_prob(odds):
    return pl.when(odds < 0).then(-odds / (-odds + 100)).otherwise(100 / (odds + 100))


TRAIN, TEST = (2006, 2019), (2020, 2025)
weights, sigma = fit_margin_model(data, TRAIN)

scored = predict(data, weights, sigma).with_columns(
    imp_home=american_to_prob(pl.col("home_moneyline")),
    imp_away=american_to_prob(pl.col("away_moneyline")),
).with_columns(mkt_p_home=pl.col("imp_home") / (pl.col("imp_home") + pl.col("imp_away")))

test = decided(scored.filter(pl.col("season").is_between(*TEST) & pl.col("mkt_p_home").is_not_null()))
y, n = test["home_won"], test.height


def score_row(name, p, show_accuracy=True):
    return {"model": name, "games": n, "log_loss": round(log_loss(p, y), 4), "brier": round(brier(p, y), 4),
            "accuracy": round(accuracy(p, y), 3) if show_accuracy else None}


pl.DataFrame([
    score_row("Coin flip (50%)", pl.Series([0.5] * n), show_accuracy=False),
    score_row("Elo", test["elo_p_home"]),
    score_row("Elo + quarterbacks + rest", test["model_p_home"]),
    score_row("Betting market (vig removed)", test["mkt_p_home"]),
])

### Season by season

Log loss for each test season. A model that's truly better should win most seasons, not just one lucky year.

In [ ]:
def season_ll(col):
    return (-(pl.when(pl.col("home_won") == 1).then(pl.col(col).log()).otherwise((1 - pl.col(col)).log()))).mean().round(4)


test.group_by("season").agg(
    pl.len().alias("games"),
    season_ll("elo_p_home").alias("elo"),
    season_ll("model_p_home").alias("new_model"),
    season_ll("mkt_p_home").alias("market"),
).sort("season")

### Calibration and spreads

**Calibration:** favorites the new model gives 70% should win about 70% of the time.

**Spreads:** the average miss is how far each predicted margin landed from the real one. Against the spread, each model bets its side whenever its spread differs from the market's; at standard −110 odds, break-even is **52.4%**.

In [ ]:
LABELS = ["50–60%", "60–70%", "70–80%", "80–90%", "90%+"]

fav = test.with_columns(
    p_fav=pl.max_horizontal(pl.col("model_p_home"), 1 - pl.col("model_p_home")),
    fav_won=pl.when(pl.col("model_p_home") >= 0.5).then(pl.col("home_won")).otherwise(1 - pl.col("home_won")),
).with_columns(
    confidence=pl.when(pl.col("p_fav") <= 0.6).then(pl.lit(LABELS[0]))
    .when(pl.col("p_fav") <= 0.7).then(pl.lit(LABELS[1]))
    .when(pl.col("p_fav") <= 0.8).then(pl.lit(LABELS[2]))
    .when(pl.col("p_fav") <= 0.9).then(pl.lit(LABELS[3]))
    .otherwise(pl.lit(LABELS[4]))
)
display(
    pl.DataFrame({"confidence": LABELS}).join(
        fav.group_by("confidence").agg(
            pl.len().alias("games"),
            pl.col("p_fav").mean().round(3).alias("predicted"),
            pl.col("fav_won").mean().round(3).alias("actual"),
        ),
        on="confidence", how="left",
    )
)

spreads = scored.filter(
    pl.col("season").is_between(*TEST) & pl.col("result").is_not_null() & pl.col("spread_line").is_not_null()
)


def spread_row(name, col):
    miss = spreads.select((pl.col(col) - pl.col("result")).abs().mean()).item()
    if col == "spread_line":
        return {"model": name, "average_miss": round(miss, 2)}
    bets = spreads.with_columns(
        gap=pl.col(col) - pl.col("spread_line"),        # positive = model likes the home team more than the market does
        vs_line=pl.col("result") - pl.col("spread_line"),  # positive = home team covered
    ).filter((pl.col("gap") != 0) & (pl.col("vs_line") != 0)).with_columns(
        won=(pl.col("gap") > 0) == (pl.col("vs_line") > 0)
    )
    big = bets.filter(pl.col("gap").abs() >= 3)
    return {"model": name, "average_miss": round(miss, 2),
            "ats_win_rate": round(bets["won"].mean(), 3),
            "ats_win_rate_3plus_gap": round(big["won"].mean(), 3), "bets_3plus_gap": big.height}


pl.DataFrame([
    spread_row("Elo", "elo_spread"),
    spread_row("Elo + quarterbacks + rest", "model_spread"),
    spread_row("Betting market", "spread_line"),
])

### What the results mean

- **The new model beats Elo.** Log loss drops from about 0.637 to 0.628, it picks the winner about 1 point more often, and it wins 5 of the 6 test seasons. Knowing who's starting at quarterback is worth something real.
- **The market is still better.** The new model closes about a quarter of the gap between Elo and the market. The market also prices in injuries beyond the quarterback, weather, matchups and sharp bettors' opinions.
- **Its percentages stay honest.** In the calibration table, its favorites win about as often as it says.
- **It still isn't a betting edge.** Against the spread it wins roughly half its bets, below the 52.4% needed to profit. Better probabilities don't automatically mean beating the line.

**Caveat:** in this test the model knows who actually started each game. Live picks use nflverse's projected starter (or your override), which is occasionally wrong if a quarterback is ruled out late.

## Step 6: The totals model (over/unders)

The totals model predicts **combined points** for each game. It uses four inputs, all known before kickoff:

| Input | What it means |
|---|---|
| `base_total` | Each team's scoring offense and defense, rated in points like Elo: after every game, ratings move toward what actually happened |
| `qb_sum` | Both starting quarterbacks' ratings added together (from Step 3) |
| `epa_sum` | Both offenses' and both defenses' efficiency, in EPA per play, from recent games |
| `pace` | How many plays both teams usually run and allow, which sets how many chances there are to score |

Like the margin model, it's a linear regression trained on past games. For the over/under, it then looks at how far its total is from the betting line, and how often games with that gap actually went over in past seasons. That keeps its over chance honest instead of overconfident.

**What was tested on the 2016–2019 practice seasons** (trained on 2006–2015):
- Scoring ratings alone missed the real total by 10.92 points on average. Adding quarterbacks cut that to 10.82, and efficiency and pace to 10.77. The market's line missed by 10.64.
- **Weather didn't help.** Dome vs. outdoors and cold months added nothing. Even the actual game-day wind and temperature, which wouldn't be known in advance, didn't improve it, so weather is left out.

In [ ]:
from sklearn.linear_model import LogisticRegression

SCORE_SETTINGS = {"k": 0.15, "regress": 0.25, "home_field": 1.0, "avg_rate": 0.005}   # chosen on 2016–2019
EFFICIENCY_SETTINGS = {"decay": 0.85, "prior_games": 2, "regress": 0.5}               # chosen on 2016–2019
TOTAL_FEATURES = ["base_total", "qb_sum", "epa_sum", "pace"]


def score_ratings(games, k=0.15, regress=0.25, home_field=1.0, avg_rate=0.005, start_avg=20.5):
    """Each team's scoring offense and defense, in points vs. the league average, from earlier games only."""
    off, dfn, avg, season, rows = {}, {}, start_avg, None, []
    for g in games.iter_rows(named=True):
        if g["season"] != season:                       # new season: shrink every rating toward average
            season = g["season"]
            off = {t: v * (1 - regress) for t, v in off.items()}
            dfn = {t: v * (1 - regress) for t, v in dfn.items()}
        h, a = g["home_team"], g["away_team"]
        edge = home_field if g["location"] == "Home" else 0.0
        exp_home = avg + edge + off.get(h, 0) + dfn.get(a, 0)
        exp_away = avg - edge + off.get(a, 0) + dfn.get(h, 0)
        rows.append({"game_id": g["game_id"], "base_total": exp_home + exp_away, "league_avg": avg})
        if g["result"] is None:
            continue
        miss_home, miss_away = g["home_score"] - exp_home, g["away_score"] - exp_away
        off[h] = off.get(h, 0) + k * miss_home
        dfn[a] = dfn.get(a, 0) + k * miss_home
        off[a] = off.get(a, 0) + k * miss_away
        dfn[h] = dfn.get(h, 0) + k * miss_away
        avg += avg_rate * ((g["home_score"] + g["away_score"]) / 2 - avg)
    return pl.DataFrame(rows)


# Each team's offense in each game: EPA and plays (passes, sacks, runs)
team_games = (
    player_stats.with_columns(pl.col("team").replace(RELOCATED))
    .group_by("game_id", "team")
    .agg(
        epa=(pl.col("passing_epa").fill_null(0) + pl.col("rushing_epa").fill_null(0)).sum(),
        plays=(pl.col("attempts").fill_null(0) + pl.col("sacks_suffered").fill_null(0) + pl.col("carries").fill_null(0)).sum(),
    )
)
team_game_stats = {(r["game_id"], r["team"]): (r["epa"], r["plays"]) for r in team_games.iter_rows(named=True)}


def efficiency_ratings(games, decay=0.85, prior_games=2, regress=0.5, league_plays=63.0):
    """Each team's offense and defense EPA per play, and plays per game, from earlier games only."""
    sums, season, rows = {}, None, []

    def value(team, kind, prior):
        s, n = sums.get((team, kind), (0.0, 0.0))
        return (s + prior * prior_games) / (n + prior_games)

    for g in games.iter_rows(named=True):
        if g["season"] != season:
            season = g["season"]
            sums = {key: (s * (1 - regress), n * (1 - regress)) for key, (s, n) in sums.items()}
        row = {"game_id": g["game_id"]}
        for side in ("home", "away"):
            team = g[f"{side}_team"]
            row[f"{side}_off_epa"] = value(team, "off_epa", 0.0)
            row[f"{side}_def_epa"] = value(team, "def_epa", 0.0)
            row[f"{side}_off_plays"] = value(team, "off_plays", league_plays)
            row[f"{side}_def_plays"] = value(team, "def_plays", league_plays)
        rows.append(row)
        if g["result"] is None:
            continue
        for team, opp in ((g["home_team"], g["away_team"]), (g["away_team"], g["home_team"])):
            epa, plays = team_game_stats.get((g["game_id"], team), (0.0, 0))
            if plays <= 0:
                continue
            for key, x in (((team, "off_epa"), epa / plays), ((opp, "def_epa"), epa / plays),
                           ((team, "off_plays"), plays), ((opp, "def_plays"), plays)):
                s, n = sums.get(key, (0.0, 0.0))
                sums[key] = (decay * s + x, decay * n + 1)
    return pl.DataFrame(rows)


tdata = (
    data.join(score_ratings(games, **SCORE_SETTINGS), on="game_id")
    .join(efficiency_ratings(games, **EFFICIENCY_SETTINGS), on="game_id")
    .with_columns(
        qb_sum=pl.col("home_qb_rating") + pl.col("away_qb_rating"),
        epa_sum=pl.col("home_off_epa") + pl.col("away_def_epa") + pl.col("away_off_epa") + pl.col("home_def_epa"),
        pace=(pl.col("home_off_plays") + pl.col("away_def_plays") + pl.col("away_off_plays") + pl.col("home_def_plays")) / 2,
    )
)


def fit_total_model(tdata, seasons):
    """Learn one weight per input (plus a constant) for combined points."""
    train = tdata.filter(pl.col("season").is_between(*seasons) & pl.col("total").is_not_null())
    X = np.column_stack([train.select(TOTAL_FEATURES).to_numpy(), np.ones(train.height)])
    weights, *_ = np.linalg.lstsq(X, train["total"].to_numpy().astype(float), rcond=None)
    return weights


def predict_total(tdata, weights):
    X = np.column_stack([tdata.select(TOTAL_FEATURES).to_numpy(), np.ones(tdata.height)])
    return tdata.with_columns(model_total=pl.Series(X @ weights))


def fit_over_chance(scored, seasons):
    """How often games went over, given how far the model's total was from the betting line."""
    t = scored.filter(
        pl.col("season").is_between(*seasons) & pl.col("total").is_not_null()
        & pl.col("total_line").is_not_null() & (pl.col("total") != pl.col("total_line"))
    )
    gap = (t["model_total"] - t["total_line"]).to_numpy().reshape(-1, 1)
    went_over = (t["total"] > t["total_line"]).to_numpy().astype(int)
    return LogisticRegression(C=1e6).fit(gap, went_over)


def add_over_chance(scored, over_model):
    line = scored["total_line"].to_numpy()
    gap = np.nan_to_num(scored["model_total"].to_numpy() - line).reshape(-1, 1)
    p = over_model.predict_proba(gap)[:, 1]
    p[np.isnan(line)] = np.nan
    return scored.with_columns(model_p_over=pl.Series(p, nan_to_null=True))

### The totals test, 2020–2025

Trained once on 2006–2019, then scored on 2020–2025. **Average miss** is how far each prediction landed from the real combined score; lower is better. **League average only** is the simplest possible guess, the same total for every game.

For over/unders, the model takes the over when its total is above the line and the under when it's below. Pushes don't count, and break-even at −110 is **52.4%**.

In [ ]:
total_weights = fit_total_model(tdata, TRAIN)
tscored = predict_total(tdata, total_weights)
over_model = fit_over_chance(tscored, TRAIN)
tscored = add_over_chance(tscored, over_model).with_columns(
    imp_over=american_to_prob(pl.col("over_odds")),
    imp_under=american_to_prob(pl.col("under_odds")),
).with_columns(mkt_p_over=pl.col("imp_over") / (pl.col("imp_over") + pl.col("imp_under")))

ttest = tscored.filter(pl.col("season").is_between(*TEST) & pl.col("total").is_not_null() & pl.col("total_line").is_not_null())


def average_miss(expr):
    return round(ttest.select((expr - pl.col("total")).abs().mean()).item(), 2)


display(pl.DataFrame([
    {"prediction": "League average only", "games": ttest.height, "average_miss": average_miss(2 * pl.col("league_avg"))},
    {"prediction": "Totals model", "games": ttest.height, "average_miss": average_miss(pl.col("model_total"))},
    {"prediction": "Betting market line", "games": ttest.height, "average_miss": average_miss(pl.col("total_line"))},
]))

ou = ttest.filter(pl.col("total") != pl.col("total_line")).with_columns(
    went_over=(pl.col("total") > pl.col("total_line")).cast(pl.Float64),
    gap=pl.col("model_total") - pl.col("total_line"),            # positive = model leans over
)
bets = ou.filter(pl.col("gap") != 0).with_columns(won=(pl.col("gap") > 0) == (pl.col("went_over") == 1))
big = bets.filter(pl.col("gap").abs() >= 3)
odds = ou.filter(pl.col("mkt_p_over").is_not_null())
display(pl.DataFrame([
    {"measure": "Over/under picks won", "value": f"{bets['won'].mean():.1%} of {bets.height}"},
    {"measure": "...when 3+ points from the line", "value": f"{big['won'].mean():.1%} of {big.height}"},
    {"measure": "Over chance log loss: model", "value": f"{log_loss(ou['model_p_over'], ou['went_over']):.4f}"},
    {"measure": "Over chance log loss: always 50%", "value": f"{log_loss(pl.Series([0.5] * ou.height), ou['went_over']):.4f}"},
    {"measure": "Over chance log loss: market", "value": f"{log_loss(odds['mkt_p_over'], odds['went_over']):.4f}"},
]))

ttest.group_by("season").agg(
    pl.len().alias("games"),
    (pl.col("model_total") - pl.col("total")).abs().mean().round(2).alias("model_miss"),
    (pl.col("total_line") - pl.col("total")).abs().mean().round(2).alias("market_miss"),
).sort("season")

### What the totals results mean

- **It predicts scoring, but the market predicts it better.** The model misses the real total by about 10.5 points, versus about 10.9 for a league-average guess and 10.3 for the market's line. The market is closer in every test season.
- **Its over/under leans are a coin flip.** They win about half the time, below the 52.4% needed to profit. That's why its over chance stays close to 50%: it's calibrated not to claim more than it knows.
- **It still earns its place in the app.** Paired with the margin model, it gives every game a projected final score, and the pick log will show over a full season whether it improves.

## Step 7: This week's games

For live picks, both models are retrained on every complete season from 2006 on, the same recipes tested above with a few more years of data.

**Projected score** comes from both models together: the home team's points are half the total plus half the margin, and the away team's are half the total minus half the margin.

**Reading the game cards**
- **Model**: who the model expects to win, and by how much. **Vegas**: the sportsbook spread, written the usual way, where the favorite gets a minus sign. *KC -3.5* means the sportsbooks have the Chiefs favored by 3.5.
- **Win chance**: the favorite's chance to win according to the model, with the sportsbooks' chance in parentheses.
- **Projected score** and **Total**: the model's expected score and combined points, next to the sportsbook over/under. Scores show one decimal so they always add up to the total and differ by the spread.
- **Model vs Vegas**: which side of the sportsbook lines the model leans to, and by how many points. Leans are usually small, and the tests show they don't beat the sportsbooks, so they're not betting advice.
- **Quarterbacks**: the starters the model is using. If one is wrong, set `QB_OVERRIDES` in Step 3.

At the top, a short list shows the games where the model and Vegas disagree most. To also see the full table of numbers, set `SHOW_NUMBERS_TABLE = True` in the cell below.

In [ ]:
from datetime import datetime

SHOW_NUMBERS_TABLE = False   # True = also show the full table of numbers under the game cards

current_season = games.filter(pl.col("result").is_null())["season"].min() or games["season"].max()
LIVE_SEASONS = (2006, current_season - 1)

live_weights, live_sigma = fit_margin_model(data, LIVE_SEASONS)
live_total_weights = fit_total_model(tdata, LIVE_SEASONS)
live_totals = predict_total(tdata, live_total_weights)
live_totals = add_over_chance(live_totals, fit_over_chance(live_totals, LIVE_SEASONS))

live = (
    predict(data, live_weights, live_sigma)
    .join(live_totals.select("game_id", "model_total", "model_p_over"), on="game_id")
    .with_columns(
        imp_home=american_to_prob(pl.col("home_moneyline")),
        imp_away=american_to_prob(pl.col("away_moneyline")),
        imp_over=american_to_prob(pl.col("over_odds")),
        imp_under=american_to_prob(pl.col("under_odds")),
    )
    .with_columns(
        mkt_p_home=pl.col("imp_home") / (pl.col("imp_home") + pl.col("imp_away")),
        mkt_p_over=pl.col("imp_over") / (pl.col("imp_over") + pl.col("imp_under")),
    )
)

w = dict(zip(FEATURES, live_weights))
tw = dict(zip(TOTAL_FEATURES, live_total_weights))
print(f"Trained on {LIVE_SEASONS[0]}–{LIVE_SEASONS[1]}. What each input is worth, in points:")
print("Margin model")
print(f"  1 point of Elo gap:                          {w['elo_pts']:.2f}")
print(f"  Home field:                                  {w['home']:.2f}")
print(f"  Each extra day of rest (up to 7):            {w['rest_diff']:.2f}")
print(f"  Starter rated 0.10 better:                   {0.1 * w['qb_diff']:.2f}")
print(f"  ...and if he's replacing a 0.10 worse QB:    {0.1 * w['qb_change_diff']:.2f} more")
print("Totals model")
print(f"  1 point of combined scoring rating:          {tw['base_total']:.2f}")
print(f"  Both starters 0.10 better combined:          {0.1 * tw['qb_sum']:.2f}")
print(f"  Combined efficiency 0.10 EPA/play better:    {0.1 * tw['epa_sum']:.2f}")
print(f"  Each extra expected play:                    {tw['pace']:.2f}")


def qb_name(qb_id):
    return qb_names.get(qb_id) or (qb_id or "").replace("new: ", "")


def signed(x):
    return "PK" if x == 0 else f"{x:+g}"


def game_card(g, when, home, away, extra=""):
    """One game in plain words. Spreads are stored from the home team's side; here they're written the way sportsbooks do."""
    ms, ml, mt, tl = g["model_spread"], g["spread_line"], g["model_total"], g["total_line"]
    pick = f"{home if ms > 0 else away} by {abs(ms):.1f}" if abs(ms) >= 0.05 else "even"
    vegas = "no line yet" if ml is None else "pick'em" if ml == 0 else f"{home if ml > 0 else away} {-abs(ml):g}"
    p = g["model_p_home"]
    fav, chance = (home, p) if p >= 0.5 else (away, 1 - p)
    win = f"{fav} {chance:.0%}"
    if g["mkt_p_home"] is not None:
        win += f" (Vegas {g['mkt_p_home'] if fav == home else 1 - g['mkt_p_home']:.0%})"
    lines = [f"{when} · {away} at {home}",
             f"  Model: {pick}   Vegas: {vegas}",
             f"  Win chance: {win}",
             f"  Projected score: {away} {(mt - ms) / 2:.1f}, {home} {(mt + ms) / 2:.1f}   "
             f"Total: {mt:.1f}" + (f" (Vegas {tl:g})" if tl is not None else "")]
    leans = []
    if ml is not None and abs(ms - ml) >= 0.05:
        side = home if ms > ml else away
        leans.append(f"{side} {signed(-ml if side == home else ml)} by {abs(ms - ml):.1f}")
    if tl is not None and abs(mt - tl) >= 0.05:
        leans.append(f"{'Over' if mt > tl else 'Under'} {tl:g} by {abs(mt - tl):.1f}")
    if leans:
        lines.append("  Model vs Vegas: leans " + " and ".join(leans) + " points")
    if extra:
        lines.append("  " + extra)
    return "\n".join(lines)


def disagreements(week, home_col, away_col, n=5):
    """The games where the model and Vegas disagree most, on the spread and on the total."""
    w = week.filter(pl.col("spread_line").is_not_null()).with_columns(gap=(pl.col("model_spread") - pl.col("spread_line")).abs()).sort("gap", descending=True).head(n)
    t = week.filter(pl.col("total_line").is_not_null()).with_columns(gap=(pl.col("model_total") - pl.col("total_line")).abs()).sort("gap", descending=True).head(n)
    out = ["Where the model and Vegas disagree most. The tests show leans like these win only about half the time, "
           "below the 52.4% needed to profit, so they're not betting advice."]
    for g in w.iter_rows(named=True):
        side = g[home_col] if g["model_spread"] > g["spread_line"] else g[away_col]
        line = -g["spread_line"] if side == g[home_col] else g["spread_line"]
        out.append(f"  Spread: {g[away_col]} at {g[home_col]}: model leans {side} {signed(line)} by {g['gap']:.1f} points")
    for g in t.iter_rows(named=True):
        out.append(f"  Total: {g[away_col]} at {g[home_col]}: model leans {'Over' if g['model_total'] > g['total_line'] else 'Under'} "
                   f"{g['total_line']:g} by {g['gap']:.1f} points")
    return "\n".join(out)


upcoming = live.filter(pl.col("result").is_null() & pl.col("mkt_p_home").is_not_null()).sort("gameday", "gametime")
if upcoming.is_empty():
    print("\nNo upcoming games have betting lines yet.")
else:
    nxt = upcoming.row(0, named=True)
    week = upcoming.filter((pl.col("season") == nxt["season"]) & (pl.col("week") == nxt["week"])).with_columns(
        away_qb=pl.col("away_qb").map_elements(qb_name, return_dtype=pl.String),
        home_qb=pl.col("home_qb").map_elements(qb_name, return_dtype=pl.String),
    )
    print(f"\n{nxt['season']} season, week {nxt['week']}: {week.height} games. Times are Eastern.\n")
    print(disagreements(week, "home_team", "away_team") + "\n")
    for g in week.iter_rows(named=True):
        when = datetime.strptime(f"{g['gameday']} {g['gametime'] or '13:00'}", "%Y-%m-%d %H:%M").strftime("%a %b %-d, %-I:%M %p")
        print(game_card(g, when, g["home_team"], g["away_team"], extra=f"Quarterbacks: {g['away_qb']} ({g['away_team']}) vs. {g['home_qb']} ({g['home_team']})") + "\n")
    if SHOW_NUMBERS_TABLE:
        display(week.select(
            "gameday", "away_team", "home_team", "away_qb", "home_qb",
            elo_spread=pl.col("elo_spread").round(1), model_spread=pl.col("model_spread").round(1), market_spread=pl.col("spread_line"),
            model_home_win=pl.col("model_p_home").round(3), market_home_win=pl.col("mkt_p_home").round(3),
            model_total=pl.col("model_total").round(1), market_total=pl.col("total_line"),
            model_over_chance=pl.col("model_p_over").round(3), market_over_chance=pl.col("mkt_p_over").round(3),
        ))

## Step 8: Save this week's picks

Same pick log as before, `nfl_pick_log.csv` in your Google Drive, now with **three models**: Elo and the margin model for winners and spreads, and the totals model for over/unders. Each run:

1. Saves every model's numbers for next week's games that haven't kicked off, with the time you ran it and each team's starting quarterback.
2. Keeps the first pick for each game and model. Running it again later never changes a logged pick.
3. Fills in final scores and grades each model separately.

Picks already in your log from earlier versions are kept, and nothing is logged twice. **Don't run notebooks older than version 5 again**, because their log format is older.

The first time, Colab asks for permission to use your Google Drive. Allow it. All times are Eastern, the time zone nflverse uses for kickoffs.

In [ ]:
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

EASTERN = ZoneInfo("America/New_York")
ELO_NAME = f"elo_v1 (k={ELO_SETTINGS['k']}, home_field={ELO_SETTINGS['home_field']}, regress={ELO_SETTINGS['regress']})"
MODELS = {
    # log name: {log column: column in `live`}
    ELO_NAME: {"model_p_home": "elo_p_home", "model_spread": "elo_spread"},
    "elo_qb_rest_v1": {"model_p_home": "model_p_home", "model_spread": "model_spread"},
    "totals_v1": {"model_total": "model_total", "model_p_over": "model_p_over"},
}
MODEL_COLUMNS = {"model_p_home": 3, "model_spread": 1, "model_total": 1, "model_p_over": 3}   # column: decimals

# In Colab the log goes to your Google Drive; anywhere else it goes next to this notebook
try:
    from google.colab import drive
    drive.mount("/content/drive")
    LOG_PATH = Path("/content/drive/MyDrive/nfl_pick_log.csv")
except ImportError:
    LOG_PATH = Path("nfl_pick_log.csv")

PICK_COLUMNS = {
    "logged_at_et": pl.String, "model": pl.String,
    "season": pl.Int32, "week": pl.Int32, "game_id": pl.String, "kickoff_et": pl.String,
    "away_team": pl.String, "home_team": pl.String, "away_qb": pl.String, "home_qb": pl.String,
    "model_p_home": pl.Float64, "market_p_home": pl.Float64,
    "model_spread": pl.Float64, "market_spread": pl.Float64,
    "model_total": pl.Float64, "market_total": pl.Float64,
    "model_p_over": pl.Float64, "market_p_over": pl.Float64,
    "home_moneyline": pl.Int32, "away_moneyline": pl.Int32,
}
OLD_NAMES = {"elo_p_home": "model_p_home", "elo_spread": "model_spread"}   # v2 log column names


def read_log():
    if not LOG_PATH.exists():
        return pl.DataFrame(schema=PICK_COLUMNS)
    log = pl.read_csv(LOG_PATH, infer_schema_length=0)       # read as text, then fix the types
    log = log.rename({k: v for k, v in OLD_NAMES.items() if k in log.columns})
    log = log.with_columns([pl.lit(None, pl.String).alias(c) for c in PICK_COLUMNS if c not in log.columns])
    return log.select([pl.col(c).cast(t) for c, t in PICK_COLUMNS.items()])


def next_week_picks(live, now):
    """Every model's numbers for the next week's games that haven't kicked off."""
    upcoming = live.with_columns(
        kickoff=pl.concat_str([pl.col("gameday"), pl.lit(" "), pl.col("gametime").fill_null("00:00")])
        .str.to_datetime("%Y-%m-%d %H:%M")
        .dt.replace_time_zone("America/New_York")
    ).filter(pl.col("result").is_null() & pl.col("mkt_p_home").is_not_null() & (pl.col("kickoff") > now))
    if upcoming.is_empty():
        return pl.DataFrame(schema=PICK_COLUMNS)
    first = upcoming.sort("kickoff").row(0, named=True)
    week = upcoming.filter((pl.col("season") == first["season"]) & (pl.col("week") == first["week"])).sort("kickoff", "game_id")
    picks = []
    for name, cols in MODELS.items():
        model_values = {
            c: (pl.col(cols[c]).round(d) if c in cols else pl.lit(None, pl.Float64)) for c, d in MODEL_COLUMNS.items()
        }
        picks.append(week.select(
            logged_at_et=pl.lit(now.astimezone(EASTERN).strftime("%Y-%m-%d %H:%M")),
            model=pl.lit(name),
            season=pl.col("season"), week=pl.col("week"), game_id=pl.col("game_id"),
            kickoff_et=pl.col("kickoff").dt.strftime("%Y-%m-%d %H:%M"),
            away_team=pl.col("away_team"), home_team=pl.col("home_team"),
            away_qb=pl.col("away_qb").map_elements(qb_name, return_dtype=pl.String),
            home_qb=pl.col("home_qb").map_elements(qb_name, return_dtype=pl.String),
            market_p_home=pl.col("mkt_p_home").round(3),
            market_spread=pl.col("spread_line"),
            market_total=pl.col("total_line"),
            market_p_over=pl.col("mkt_p_over").round(3),
            home_moneyline=pl.col("home_moneyline"), away_moneyline=pl.col("away_moneyline"),
            **model_values,
        ).select(list(PICK_COLUMNS)).cast(PICK_COLUMNS))
    return pl.concat(picks)


def update_log(live, now=None):
    """Add new pre-kickoff picks, fill in final scores, and save the log."""
    now = now or datetime.now(EASTERN)
    log = read_log()
    # The first pick for each game and model is the one that counts: logged picks are never replaced
    added = next_week_picks(live, now).join(log, on=["game_id", "model"], how="anti")
    log = pl.concat([log, added])
    finals = live.filter(pl.col("result").is_not_null()).select(
        "game_id", "away_score", "home_score", "result",
        total_points=pl.col("total"),
        closing_spread=pl.col("spread_line"),
        closing_total=pl.col("total_line"),
    )
    log = log.join(finals, on="game_id", how="left").sort("kickoff_et", "model", "game_id")
    log.write_csv(LOG_PATH)
    return log, added


def log_scorecard(log):
    """Grade each model on its logged games that have final scores."""
    graded = log.filter(pl.col("result").is_not_null())
    if graded.is_empty():
        print("No logged games have finished yet. Run this again after the week's games to grade them.")
        return
    print(f"Graded picks from {graded['kickoff_et'].min()[:10]} to {graded['kickoff_et'].max()[:10]}. Break-even at -110 is 52.4%.")
    print("'Line moved toward' counts games where the line moved toward the model's side after logging.\n")

    sides = graded.filter(pl.col("model_p_home").is_not_null()).with_columns(
        home_won=pl.when(pl.col("result") > 0).then(1.0).when(pl.col("result") < 0).then(0.0).otherwise(0.5),
        gap=pl.col("model_spread") - pl.col("market_spread"),             # positive = model likes the home team more
        vs_line=pl.col("result") - pl.col("market_spread"),               # positive = home team covered
        line_move=pl.col("closing_spread") - pl.col("market_spread"),     # positive = line moved toward home
    )
    totals = graded.filter(pl.col("model_total").is_not_null()).with_columns(
        gap=pl.col("model_total") - pl.col("market_total"),               # positive = model leans over
        vs_line=pl.col("total_points") - pl.col("market_total"),          # positive = went over
        line_move=pl.col("closing_total") - pl.col("market_total"),       # positive = line moved up
    )

    def record(g):
        bets = g.filter(pl.col("gap") != 0)
        pushes = bets.filter(pl.col("vs_line") == 0).height
        wins = bets.filter((pl.col("vs_line") != 0) & ((pl.col("gap") > 0) == (pl.col("vs_line") > 0))).height
        losses = bets.height - wins - pushes
        moves = bets.filter(pl.col("line_move").is_not_null()).with_columns(toward=pl.col("line_move") * pl.col("gap").sign())
        return {
            "record": f"{wins}-{losses}-{pushes}",
            "win_rate": round(wins / (wins + losses), 3) if wins + losses else None,
            "line_moved_toward": moves.filter(pl.col("toward") > 0).height,
            "line_moved_away": moves.filter(pl.col("toward") < 0).height,
        }

    if not sides.is_empty():
        print("Winners and spreads:")
        display(pl.DataFrame([
            {"model": name, "graded_games": g.height,
             "model_log_loss": round(log_loss(g["model_p_home"], g["home_won"]), 4),
             "market_log_loss": round(log_loss(g["market_p_home"], g["home_won"]), 4),
             **record(g)}
            for name in sides["model"].unique(maintain_order=True)
            for g in [sides.filter(pl.col("model") == name)]
        ]))
    if not totals.is_empty():
        print("Totals:")
        display(pl.DataFrame([
            {"model": name, "graded_games": g.height,
             "model_miss": round(g.select((pl.col("model_total") - pl.col("total_points")).abs().mean()).item(), 2),
             "market_miss": round(g.select((pl.col("market_total") - pl.col("total_points")).abs().mean()).item(), 2),
             **record(g)}
            for name in totals["model"].unique(maintain_order=True)
            for g in [totals.filter(pl.col("model") == name)]
        ]))

In [ ]:
log, added = update_log(live)
print(f"Pick log: {LOG_PATH} ({log.height} picks logged so far)\n")

if added.is_empty():
    print("Nothing new to log. This week's games are already in the log, or none have betting lines yet.\n")
else:
    for name in added["model"].unique(maintain_order=True):
        a = added.filter(pl.col("model") == name)
        print(f"Logged {a.height} games for {name}, {a['season'][0]} season, week {a['week'][0]}")
    print()

log_scorecard(log)

## Step 9: The player props model

This model projects four common player props:

| Prop | Who gets one |
|---|---|
| Passing yards | Each team's starting quarterback |
| Rushing yards | Running backs and quarterbacks who average 5+ carries |
| Receiving yards | Receivers, tight ends and running backs who average 3+ targets |
| Receptions | The same players as receiving yards |

A player needs at least 3 games of history to get props.

**How a projection works**
1. **Team volume.** For each team, it projects passes, passing yards, runs and rushing yards. It uses the team's recent games, what the opponent's defense allows, and the game script, meaning the spread and total. Teams expected to score more throw for more yards, and big favorites run more. The spread and total come from the betting market when there's a line, because Steps 5 and 6 showed the market is more accurate than our models. Our projections fill in when a line is missing.
2. **Player share.** It looks at each player's share of his team's targets or carries, plus his yards per target, catch rate or yards per carry. Recent games count more, and a player with little history starts near the typical rates for his position. Shares are split among the teammates who play, so when a teammate is out, his targets and carries go to the others.
3. **Blend.** The share-based projection is blended with the player's recent average, using weights learned from past games.
4. **Over chance.** To turn a projection into an over chance for any line, it looks at past players with similar projections and how far above or below their projections they finished.

**What was tested on the 2016–2019 practice seasons** (trained on 2006–2015):
- For team passing yards, adding the market's spread and total cut the average miss from 57.7 to 56.3 yards per game.
- Adding the quarterback's own yards per attempt barely changed the miss once the market's lines were in, since the market already prices in who's playing quarterback. It's left out.
- Splitting shares among the players who played helped rushing projections most. Receiving projections barely changed.

**Caveat:** in testing, the model knew which players actually played each game, the same way it knew the starting quarterbacks. Live projections use the active roster and the injury report instead, which can miss a late scratch.

In [ ]:
PROP_TEAM_SETTINGS = {"decay": 0.9, "prior_games": 2, "regress": 0.5}                                              # chosen on 2016–2019
PROP_PLAYER_SETTINGS = {"decay": 0.85, "prior_games": 2, "regress": 0.4, "prior_targets": 30, "prior_carries": 60}   # chosen on 2016–2019
TEAM_STATS = ["attempts", "passing_yards", "carries", "rushing_yards"]
PROP_STATS = ["passing_yards", "rushing_yards", "receiving_yards", "receptions"]
BOX_STATS = ["attempts", "passing_yards", "carries", "rushing_yards", "targets", "receptions", "receiving_yards"]
MIN_GAMES = 3   # a player needs 3 games of history to get props

# Typical rates for a player with little history, by position
POSITION_PRIORS = {
    "WR": {"yards_per_target": 8.0, "catch_rate": 0.62, "yards_per_carry": 6.0},
    "TE": {"yards_per_target": 7.0, "catch_rate": 0.70, "yards_per_carry": 4.0},
    "RB": {"yards_per_target": 5.8, "catch_rate": 0.77, "yards_per_carry": 4.2},
    "QB": {"yards_per_target": 5.0, "catch_rate": 0.60, "yards_per_carry": 4.5},
}

# Box scores: each team's totals and each player's line in every game
box = player_stats.with_columns(pl.col("team").replace(RELOCATED), *[pl.col(c).fill_null(0) for c in BOX_STATS])
team_box = {
    (r["game_id"], r["team"]): r
    for r in box.group_by("game_id", "team").agg([pl.col(c).sum() for c in BOX_STATS]).iter_rows(named=True)
}
players_by_game = {}
for r in box.filter((pl.col("attempts") + pl.col("carries") + pl.col("targets")) > 0).iter_rows(named=True):
    players_by_game.setdefault(r["game_id"], []).append(r)
starter_by_game = {r["game_id"]: (r["home_qb"], r["away_qb"]) for r in qb.iter_rows(named=True)}


def team_volume(games, decay=0.9, prior_games=2, regress=0.5, rate=0.01):
    """Each team's passes, passing yards, runs and rushing yards per game, and what its defense allows, from earlier games only."""
    sums, season, rows = {}, None, []
    league = {"attempts": 32.0, "passing_yards": 210.0, "carries": 27.0, "rushing_yards": 115.0}

    def value(team, side, stat):
        s, n = sums.get((team, side, stat), (0.0, 0.0))
        return (s + league[stat] * prior_games) / (n + prior_games)

    for g in games.iter_rows(named=True):
        if g["season"] != season:                       # new season: older games count less
            season = g["season"]
            sums = {k: (s * (1 - regress), n * (1 - regress)) for k, (s, n) in sums.items()}
        home_qb, away_qb = starter_by_game.get(g["game_id"], (None, None))
        for team, opp, is_home, starter in ((g["home_team"], g["away_team"], True, home_qb),
                                            (g["away_team"], g["home_team"], False, away_qb)):
            played = team_box.get((g["game_id"], team)) if g["result"] is not None else None
            row = {"game_id": g["game_id"], "season": g["season"], "team": team, "opp": opp, "is_home": is_home, "qb": starter}
            for stat in TEAM_STATS:
                row[f"own_{stat}"] = value(team, "offense", stat)
                row[f"opp_{stat}"] = value(opp, "defense", stat)
                row[f"actual_{stat}"] = float(played[stat]) if played else None
            rows.append(row)
        if g["result"] is None:
            continue
        for team, opp in ((g["home_team"], g["away_team"]), (g["away_team"], g["home_team"])):
            played = team_box.get((g["game_id"], team))
            if played is None:
                continue
            for stat in TEAM_STATS:
                for key in ((team, "offense", stat), (opp, "defense", stat)):
                    s, n = sums.get(key, (0.0, 0.0))
                    sums[key] = (decay * s + played[stat], decay * n + 1)
                league[stat] += rate * (played[stat] - league[stat])
    return pl.DataFrame(rows, infer_schema_length=None)


def player_snapshot(totals, position, season, last_season, prior_games=2, regress=0.4, prior_targets=30, prior_carries=60, **_):
    """A player's shares, rates and recent averages going into a game, from his decayed totals."""
    s = totals
    if last_season is not None and last_season != season:     # first game of a new season: older games count less
        s = {k: v * (1 - regress) for k, v in s.items()}
    prior = POSITION_PRIORS.get(position, POSITION_PRIORS["WR"])
    n, full = s.get("games", 0.0), s.get("full_games", 0.0)

    def rate(num, den, typical, weight):
        return (s.get(num, 0.0) + typical * weight) / (s.get(den, 0.0) + weight)

    return {
        "target_share": (s.get("target_share", 0.0) + 0.04 * prior_games) / (n + prior_games),
        "carry_share": (s.get("carry_share", 0.0) + 0.02 * prior_games) / (n + prior_games),
        "yards_per_target": rate("receiving_yards", "targets", prior["yards_per_target"], prior_targets),
        "catch_rate": rate("receptions", "targets", prior["catch_rate"], prior_targets),
        "yards_per_carry": rate("rushing_yards", "carries", prior["yards_per_carry"], prior_carries),
        # recent averages per game played (recent games count more)
        "avg_targets": s.get("g_targets", 0.0) / n if n else None,
        "avg_carries": s.get("g_carries", 0.0) / n if n else None,
        "avg_receiving_yards": s.get("g_receiving_yards", 0.0) / n if n else None,
        "avg_receptions": s.get("g_receptions", 0.0) / n if n else None,
        "avg_rushing_yards": s.get("g_rushing_yards", 0.0) / n if n else None,
        "avg_passing_yards": s.get("g_full_passing_yards", 0.0) / full if full else None,   # games with 15+ passes
    }


def player_ratings(games, decay=0.85, **settings):
    """Every player's shares and rates BEFORE each game he played, plus everyone's latest totals for next week."""
    totals, last_season, games_played, rows = {}, {}, {}, []
    for g in games.filter(pl.col("result").is_not_null()).iter_rows(named=True):
        played = players_by_game.get(g["game_id"], [])
        for r in played:
            p = r["player_id"]
            snap = player_snapshot(totals.get(p, {}), r["position_group"], g["season"], last_season.get(p), **settings)
            rows.append({"game_id": g["game_id"], "team": r["team"], "player_id": p, "player": r["player_display_name"],
                         "position": r["position_group"], "games_before": games_played.get(p, 0), **snap,
                         **{f"actual_{k}": float(r[k]) for k in PROP_STATS}})
        for r in played:
            p = r["player_id"]
            if last_season.get(p) is not None and last_season[p] != g["season"]:
                totals[p] = {k: v * (1 - settings["regress"]) for k, v in totals[p].items()}
            last_season[p] = g["season"]
            team = team_box[(g["game_id"], r["team"])]
            full_game = r["attempts"] >= 15
            update = {
                "games": 1.0,
                "target_share": r["targets"] / team["targets"] if team["targets"] > 0 else 0.0,
                "carry_share": r["carries"] / team["carries"] if team["carries"] > 0 else 0.0,
                "targets": r["targets"], "receptions": r["receptions"], "receiving_yards": r["receiving_yards"],
                "carries": r["carries"], "rushing_yards": r["rushing_yards"],
                "g_targets": r["targets"], "g_carries": r["carries"], "g_receiving_yards": r["receiving_yards"],
                "g_receptions": r["receptions"], "g_rushing_yards": r["rushing_yards"],
                "full_games": 1.0 if full_game else 0.0, "g_full_passing_yards": r["passing_yards"] if full_game else 0.0,
            }
            s = totals.setdefault(p, {})
            for k, v in update.items():
                s[k] = decay * s.get(k, 0.0) + v
            games_played[p] = games_played.get(p, 0) + 1
    return pl.DataFrame(rows, infer_schema_length=None), totals, last_season, games_played


# Game script: the market's spread and total when there is one, our own projections otherwise
game_context = live.select(
    "game_id",
    spread=pl.coalesce("spread_line", "model_spread"),
    game_total=pl.coalesce("total_line", "model_total"),
)
volume = (
    team_volume(games, **PROP_TEAM_SETTINGS)
    .join(game_context, on="game_id", how="left")
    .with_columns(team_spread=pl.when(pl.col("is_home")).then(pl.col("spread")).otherwise(-pl.col("spread")))
    .with_columns(implied_points=(pl.col("game_total") + pl.col("team_spread")) / 2)
)
prop_rows, player_totals, player_last_season, player_games = player_ratings(games, **PROP_PLAYER_SETTINGS)
TEAM_FEATURES = ["team_spread", "implied_points"]

# The four props: who gets each one, and the projection from opportunity that each one starts from
PROPS = {
    "passing_yards": {"who": pl.col("player_id") == pl.col("qb"),
                      "opportunity": pl.col("team_passing_yards")},
    "rushing_yards": {"who": pl.col("position").is_in(["RB", "QB"]) & (pl.col("avg_carries") >= 5),
                      "opportunity": pl.col("team_rushing_yards") * pl.col("carry_share_n") * pl.col("yards_per_carry")
                      / (pl.col("own_rushing_yards") / pl.col("own_carries"))},
    "receiving_yards": {"who": pl.col("position").is_in(["WR", "TE", "RB"]) & (pl.col("avg_targets") >= 3),
                        "opportunity": pl.col("team_passing_yards") * pl.col("target_share_n") * pl.col("yards_per_target")
                        / (pl.col("own_passing_yards") / pl.col("own_attempts"))},
    "receptions": {"who": pl.col("position").is_in(["WR", "TE", "RB"]) & (pl.col("avg_targets") >= 3),
                   "opportunity": pl.col("team_attempts") * pl.col("target_share_n") * pl.col("catch_rate")},
}


def fit_linear(df, features, target):
    X = np.column_stack([df.select(features).to_numpy().astype(float), np.ones(df.height)])
    weights, *_ = np.linalg.lstsq(X, df[target].to_numpy().astype(float), rcond=None)
    return weights


def apply_linear(df, features, weights):
    return np.column_stack([df.select(features).to_numpy().astype(float), np.ones(df.height)]) @ weights


def fit_team_models(volume, seasons):
    """For each team stat: weights for the team's own average, the opponent's, and the game script."""
    train = volume.filter(pl.col("season").is_between(*seasons) & pl.col("actual_attempts").is_not_null() & pl.col("team_spread").is_not_null())
    return {stat: fit_linear(train, [f"own_{stat}", f"opp_{stat}"] + TEAM_FEATURES, f"actual_{stat}") for stat in TEAM_STATS}


def add_team_projections(volume, team_models):
    return volume.with_columns([
        pl.Series(f"team_{stat}", apply_linear(volume, [f"own_{stat}", f"opp_{stat}"] + TEAM_FEATURES, w))
        for stat, w in team_models.items()
    ])


def prop_table(players, team_projections):
    """One row per player and prop. Shares are split among the teammates who play, so a missing teammate's share goes to the rest."""
    cols = ["game_id", "season", "team", "opp", "qb", *[f"own_{s}" for s in TEAM_STATS], *[f"team_{s}" for s in TEAM_STATS]]
    d = players.join(team_projections.select(cols), on=["game_id", "team"]).with_columns(
        target_share_n=pl.col("target_share") / pl.col("target_share").sum().over("game_id", "team"),
        carry_share_n=pl.col("carry_share") / pl.col("carry_share").sum().over("game_id", "team"),
    )
    out = []
    for stat, spec in PROPS.items():
        out.append(d.filter(spec["who"] & (pl.col("games_before") >= MIN_GAMES)).select(
            "game_id", "season", "team", "opp", "player_id", "player", "position",
            stat=pl.lit(stat),
            opportunity=spec["opportunity"],
            recent_avg=pl.coalesce(f"avg_{stat}", spec["opportunity"]),
            actual=(pl.col(f"actual_{stat}") if f"actual_{stat}" in d.columns else pl.lit(None)).cast(pl.Float64),
        ))
    return pl.concat(out)


def fit_prop_models(table, seasons):
    """For each prop: weights for the opportunity projection and the recent average, plus how far past players finished from their projections."""
    models = {}
    for stat in PROPS:
        train = table.filter((pl.col("stat") == stat) & pl.col("season").is_between(*seasons) & pl.col("actual").is_not_null())
        weights = fit_linear(train, ["opportunity", "recent_avg"], "actual")
        proj = apply_linear(train, ["opportunity", "recent_avg"], weights)
        edges = np.quantile(proj, np.linspace(0, 1, 11))[1:-1]        # ten groups of similar projections
        group = np.digitize(proj, edges)
        actual = train["actual"].to_numpy()
        ok = proj > 0
        ratios = [np.sort(actual[(group == k) & ok] / proj[(group == k) & ok]) for k in range(10)]
        models[stat] = {"weights": weights, "edges": edges, "ratios": ratios}
    return models


def project_props(table, models):
    """Projection (average), 50/50 line, and the middle half of likely outcomes."""
    out = []
    for stat, m in models.items():
        t = table.filter(pl.col("stat") == stat)
        proj = apply_linear(t, ["opportunity", "recent_avg"], m["weights"])
        group = np.digitize(proj, m["edges"])
        q = np.array([np.quantile(m["ratios"][k], [0.25, 0.5, 0.75]) for k in group]).reshape(-1, 3) * proj[:, None]
        out.append(t.with_columns(projection=pl.Series(proj), low=pl.Series(q[:, 0]), median=pl.Series(q[:, 1]),
                                  high=pl.Series(q[:, 2]), group=pl.Series(group, dtype=pl.Int64)))
    return pl.concat(out)


def add_over_chance(props, models, line_col="line"):
    """Share of similar past players who finished above the line, scaled to each player's projection."""
    chances = []
    for r in props.select("stat", "projection", "group", line_col).iter_rows(named=True):
        line, proj = r[line_col], r["projection"]
        if line is None or proj is None or proj <= 0:
            chances.append(None)
            continue
        ratios = models[r["stat"]]["ratios"][r["group"]]
        chances.append(1 - np.searchsorted(ratios, line / proj, side="right") / len(ratios))
    return props.with_columns(p_over=pl.Series(chances, dtype=pl.Float64))

### The props test, 2020–2025

The model is trained once on 2006–2019, then scored on every player with a prop in 2020–2025. Historical sportsbook prop lines aren't free, so the model is compared with a simple stand-in: **the player's recent average**, with recent games counting more.

- **Average miss** is how far each projection landed from what the player actually did. Lower is better.
- **Over/under at his recent average** sets a line at each player's recent average, rounded to x.5, and checks whether the model's over chances are honest. Lower log loss is better. `same_chance` gives every player the same over chance, based on how often players went over that line in past seasons. `coin_flip` says 50% every time.
- **Calibration**: props the model gives the over 60% should go over about 60% of the time.

In [ ]:
PROP_NAMES = {"passing_yards": "Passing yards", "rushing_yards": "Rushing yards",
              "receiving_yards": "Receiving yards", "receptions": "Receptions"}

team_models = fit_team_models(volume, TRAIN)
test_table = prop_table(prop_rows, add_team_projections(volume, team_models))
prop_models = fit_prop_models(test_table, TRAIN)
props_scored = project_props(test_table, prop_models)
ptest = props_scored.filter(pl.col("season").is_between(*TEST) & pl.col("actual").is_not_null())

# Average miss: the model's projection vs. the player's recent average
by_season = ptest.group_by("stat", "season").agg(
    ((pl.col("projection") - pl.col("actual")).abs().mean() < (pl.col("recent_avg") - pl.col("actual")).abs().mean()).alias("model_better")
)
miss = ptest.group_by("stat").agg(
    pl.len().alias("props"),
    (pl.col("recent_avg") - pl.col("actual")).abs().mean().round(2).alias("recent_average_miss"),
    (pl.col("projection") - pl.col("actual")).abs().mean().round(2).alias("model_miss"),
).join(by_season.group_by("stat").agg(pl.col("model_better").sum().alias("seasons_model_closer")), on="stat")
display(miss.with_columns(
    prop=pl.col("stat").replace(PROP_NAMES),
    improvement=(1 - pl.col("model_miss") / pl.col("recent_average_miss")).round(3),
    order=pl.col("stat").replace({s: str(i) for i, s in enumerate(PROP_NAMES)}),
).sort("order").select("prop", "props", "recent_average_miss", "model_miss", "improvement", "seasons_model_closer"))

# Over/under at a line set at each player's recent average
at_avg = ptest.with_columns(line=pl.col("recent_avg").floor() + 0.5)
at_avg = add_over_chance(at_avg, prop_models).with_columns(went_over=(pl.col("actual") > pl.col("line")).cast(pl.Float64))
train_rate = (
    props_scored.filter(pl.col("season").is_between(*TRAIN) & pl.col("actual").is_not_null())
    .group_by("stat").agg(train_over=(pl.col("actual") > pl.col("recent_avg").floor() + 0.5).mean())
)
at_avg = at_avg.join(train_rate, on="stat")
display(pl.DataFrame([
    {"prop": PROP_NAMES[stat], "props": g.height,
     "went_over": round(g["went_over"].mean(), 3),
     "model_log_loss": round(log_loss(g["p_over"], g["went_over"]), 4),
     "same_chance_log_loss": round(log_loss(g["train_over"], g["went_over"]), 4),
     "coin_flip_log_loss": round(log_loss(pl.Series([0.5] * g.height), g["went_over"]), 4)}
    for stat in PROP_NAMES for g in [at_avg.filter(pl.col("stat") == stat)]
]))

# Calibration: when the model gives the over a certain chance, how often it went over
OVER_LABELS = ["under 30%", "30–40%", "40–50%", "50–60%", "60–70%", "70%+"]
cal = at_avg.with_columns(
    chance=pl.when(pl.col("p_over") < 0.3).then(pl.lit(OVER_LABELS[0]))
    .when(pl.col("p_over") < 0.4).then(pl.lit(OVER_LABELS[1]))
    .when(pl.col("p_over") < 0.5).then(pl.lit(OVER_LABELS[2]))
    .when(pl.col("p_over") < 0.6).then(pl.lit(OVER_LABELS[3]))
    .when(pl.col("p_over") < 0.7).then(pl.lit(OVER_LABELS[4]))
    .otherwise(pl.lit(OVER_LABELS[5]))
)
pl.DataFrame({"model_over_chance": OVER_LABELS}).join(
    cal.group_by("chance").agg(
        pl.len().alias("props"),
        pl.col("p_over").mean().round(3).alias("predicted"),
        pl.col("went_over").mean().round(3).alias("actual"),
    ).rename({"chance": "model_over_chance"}),
    on="model_over_chance", how="left",
)

### What the props results mean

- **It beats the player's recent average for all four props in every test season, but by small margins.** The miss shrinks about 4–5% for passing and rushing yards and about 2% for receiving yards and receptions. Most of a player's week-to-week swing is luck that no projection can see coming.
- **Game script matters most for quarterbacks and running backs.** That's where knowing the spread and total helps.
- **Players finish under their recent average more often than over.** At a line set at his recent average, receivers and running backs went over only about 40% of the time, because a few big games pull an average up. That's why the 50/50 line sits below the projection for most receivers and runners. Sportsbooks know this, so their lines usually sit close to the 50/50 line.
- **Its over chances are honest.** In the calibration table, props it gave the over 60–70% went over about that often.
- **This doesn't show a betting edge.** Real sportsbook lines are much sharper than a recent average. The props log in Step 11 tracks how it does against lines you enter.

## Step 10: This week's props

Every player with a prop this week, sorted from highest projection to lowest:
- **projection**: his average outcome.
- **line_50_50**: he finishes above it half the time. Compare this with the sportsbook's line.
- **likely_range**: the middle half of outcomes. He lands in this range about half the time.

**Who's included:** each team's listed starting quarterback (change it with `QB_OVERRIDES` in Step 3), plus running backs, receivers and tight ends on the active roster who've played this season or are back from a reserve list. Anyone listed **Out** or **Doubtful** on the injury report is left out. Final injury reports come out Friday for Sunday games, so a Thursday run doesn't know about Sunday scratches yet.

**Sportsbook lines come in automatically** when the `ODDS_API_KEY` secret is set (a free key from the-odds-api.com). For each prop, the line is the one most sportsbooks offer, with the middle of their over and under prices. To save the free plan's monthly credits, each game's lines are fetched once it's within 3 days of kickoff, at most every 12 hours, and saved in `odds/nfl_prop_odds.json` next to your pick log. That's about 4 credits per game, so a Thursday run gets the Thursday game and a Sunday run gets the rest. The table then shows the line and the model's over chance at it.

**To use your own sportsbook's lines instead:** type them into `PROP_LINES` below, for example `PROP_LINES = {("Josh Allen", "passing_yards"): 245.5, ("Puka Nacua", "receptions"): 7.5}`. A line you type in wins over the automatic one. The prop names are `passing_yards`, `rushing_yards`, `receiving_yards` and `receptions`. Spell player names the way they appear in the table.

To leave out a player ruled out late, add his name to `PLAYERS_OUT`. To see just a few games, list teams in `SHOW_TEAMS`.

In [ ]:
import os
import json
import statistics
import requests
from datetime import timedelta

# Sportsbook prop lines from The Odds API (the-odds-api.com). The key comes from Colab's Secrets panel or the repository
# secret on GitHub, so it never appears in the notebook. Without one, this step uses only the lines in PROP_LINES below.
ODDS_WINDOW_HOURS = 72    # fetch lines for games that kick off within this many hours...
ODDS_REFRESH_HOURS = 12   # ...unless that game's lines were fetched more recently than this
ODDS_MARKETS = {"player_pass_yds": "passing_yards", "player_rush_yds": "rushing_yards",
                "player_reception_yds": "receiving_yards", "player_receptions": "receptions"}
ODDS_URL = os.environ.get("ODDS_API_URL", "https://api.the-odds-api.com") + "/v4/sports/americanfootball_nfl"
ODDS_FILE = LOG_PATH.parent / "odds" / "nfl_prop_odds.json"     # saved lines, next to your pick log

try:
    from google.colab import userdata
    try:
        ODDS_KEY = userdata.get("ODDS_API_KEY")
    except Exception:
        ODDS_KEY = None
except ImportError:
    ODDS_KEY = os.environ.get("ODDS_API_KEY")        # GitHub Actions passes the repository secret this way
ODDS_KEY = (ODDS_KEY or "").strip() or None
SUFFIXES = {"jr", "sr", "ii", "iii", "iv", "v"}


class OddsStop(Exception):
    """The key is wrong or out of credits: stop asking for the rest of this run."""


def name_key(name):
    """A player's name for matching across data sources: "D.J. Moore Jr." and "DJ Moore" both become "dj moore"."""
    text = "".join(c if c.isalnum() or c.isspace() else " " if c == "-" else "" for c in (name or "").lower())
    return " ".join(w for w in text.split() if w not in SUFFIXES)


def to_prob(price):
    return 100 / (price + 100) if price > 0 else -price / (-price + 100)


def middle_price(prices):
    """The middle of several books' prices, worked out on their implied chances so +105 and -105 average sensibly."""
    q = statistics.median(to_prob(p) for p in prices)
    return round(-100 * q / (1 - q)) if q >= 0.5 else round(100 * (1 - q) / q)


def odds_request(path, **params):
    """One request to The Odds API. Returns the data, or None if it didn't work. A bad key or no credits stops the rest."""
    try:
        r = requests.get(f"{ODDS_URL}{path}", params={"apiKey": ODDS_KEY, **params}, timeout=30)
    except requests.RequestException as e:      # its message can include the key, so only the type is printed
        print(f"Couldn't reach The Odds API ({type(e).__name__}).")
        return None
    if r.headers.get("x-requests-remaining") is not None:
        odds_saved["credits"] = {"remaining": r.headers["x-requests-remaining"], "used": r.headers.get("x-requests-used"),
                                 "checked": datetime.now(EASTERN).isoformat(timespec="minutes")}
    if r.ok:
        return r.json()
    try:
        code = r.json().get("error_code") or str(r.status_code)
    except ValueError:
        code = str(r.status_code)
    hints = {"INVALID_KEY": "Check the ODDS_API_KEY secret.",
             "OUT_OF_USAGE_CREDITS": "This month's credits are used up. Lines come back when they reset."}
    print(f"The Odds API said no ({code}). {hints.get(code, '')}".strip())
    if code in ("INVALID_KEY", "MISSING_KEY", "DEACTIVATED_KEY", "OUT_OF_USAGE_CREDITS") or r.status_code == 401:
        raise OddsStop(code)
    return None


def book_lines(week_games, now):
    """Each player's main sportsbook line for this week's games: the line most books offer, and the middle of their
    over and under prices there. Fetches games that kick off soon; earlier fetches are reused from the saved file."""
    global odds_saved
    odds_saved = json.loads(ODDS_FILE.read_text()) if ODDS_FILE.exists() else {"games": {}}
    games_saved = odds_saved["games"]
    soon = [g for g in week_games.iter_rows(named=True)
            if g["kickoff"] - now < timedelta(hours=ODDS_WINDOW_HOURS)
            and not (games_saved.get(g["game_id"], {}).get("standard", {}).get("fetched")
                     and now - datetime.fromisoformat(games_saved[g["game_id"]]["standard"]["fetched"]) < timedelta(hours=ODDS_REFRESH_HOURS))]
    fetched = 0
    if ODDS_KEY and soon:
        nick = dict(nfl.load_teams().select("team_abbr", "team_nick").iter_rows())
        try:
            events = odds_request("/events") or []       # listing the games is free
            for g in soon:
                want = {nick.get(g["home_team"]), nick.get(g["away_team"])}
                match = [e for e in events if {e["home_team"].split()[-1], e["away_team"].split()[-1]} == want
                         and abs(datetime.fromisoformat(e["commence_time"].replace("Z", "+00:00")) - g["kickoff"]) < timedelta(hours=36)]
                if not match:
                    print(f"No sportsbook game found for {g['away_team']} at {g['home_team']}.")
                    continue
                data = odds_request(f"/events/{match[0]['id']}/odds", regions="us", oddsFormat="american", markets=",".join(ODDS_MARKETS))
                if data is None:
                    continue
                lines = {}
                for book in data.get("bookmakers", []):
                    for m in book.get("markets", []):
                        stat, sides = ODDS_MARKETS.get(m["key"]), {}
                        for o in m.get("outcomes", []):
                            if stat and o.get("point") is not None and o.get("description"):
                                sides.setdefault((o["description"], o["point"]), {})[o["name"]] = o["price"]
                        for (player, point), p in sides.items():
                            if "Over" in p and "Under" in p:
                                lines.setdefault(f"{name_key(player)}|{stat}", []).append([book["key"], point, p["Over"], p["Under"]])
                game = games_saved.setdefault(g["game_id"], {})
                game.update(event_id=match[0]["id"], kickoff=g["kickoff"].isoformat(),
                            standard={"fetched": now.isoformat(timespec="minutes"), "lines": lines})
                fetched += 1
        except OddsStop:
            pass
        cutoff = now - timedelta(days=10)                # forget games from more than 10 days ago
        odds_saved["games"] = {k: v for k, v in games_saved.items() if datetime.fromisoformat(v["kickoff"]) > cutoff}
        ODDS_FILE.parent.mkdir(parents=True, exist_ok=True)
        ODDS_FILE.write_text(json.dumps(odds_saved, indent=1, sort_keys=True))

    rows = []
    for g in week_games.iter_rows(named=True):
        for k, entries in odds_saved["games"].get(g["game_id"], {}).get("standard", {}).get("lines", {}).items():
            player, stat = k.rsplit("|", 1)
            points = [e[1] for e in entries]
            most = max(points.count(p) for p in points)
            point = statistics.median_low(sorted({p for p in points if points.count(p) == most}))
            at = [e for e in entries if e[1] == point]
            rows.append({"game_id": g["game_id"], "name_key": player, "stat": stat, "book_line": float(point),
                         "over_price": middle_price([e[2] for e in at]), "under_price": middle_price([e[3] for e in at]), "books": len(at)})
    schema = {"game_id": pl.String, "name_key": pl.String, "stat": pl.String, "book_line": pl.Float64,
              "over_price": pl.Int64, "under_price": pl.Int64, "books": pl.Int64}
    credits = odds_saved.get("credits", {})
    if not ODDS_KEY:
        print("No ODDS_API_KEY, so props only have the lines you type into PROP_LINES.")
    else:
        print(f"Sportsbook lines: fetched {fetched} of {len(soon)} games due" + (f"; {credits['remaining']} credits left this month" if credits else "")
              + f". {len({r['game_id'] for r in rows})} games have lines.")
    return pl.DataFrame(rows, schema=schema)

In [ ]:
# Late news only. Players who won't play, spelled the way nflverse writes them. Example: PLAYERS_OUT = ["Puka Nacua"]
PLAYERS_OUT = []
# Sportsbook lines you want an over chance for. Example: PROP_LINES = {("Josh Allen", "passing_yards"): 245.5}
PROP_LINES = {}
# Teams to show. Leave empty for every game. Example: SHOW_TEAMS = ["BUF", "LV"]
SHOW_TEAMS = []

live_team_models = fit_team_models(volume, LIVE_SEASONS)
live_team_proj = add_team_projections(volume, live_team_models)
live_prop_models = fit_prop_models(prop_table(prop_rows, live_team_proj), LIVE_SEASONS)

player_names = dict(box.select("player_id", "player_display_name").iter_rows())
player_positions = dict(box.sort("season", "week").select("player_id", "position_group").iter_rows())


def kickoff_times(df):
    return df.with_columns(
        kickoff=pl.concat_str([pl.col("gameday"), pl.lit(" "), pl.col("gametime").fill_null("00:00")])
        .str.to_datetime("%Y-%m-%d %H:%M")
        .dt.replace_time_zone("America/New_York")
    )


def next_week_games(now):
    upcoming = kickoff_times(games).filter(pl.col("result").is_null() & (pl.col("kickoff") > now))
    if upcoming.is_empty():
        return upcoming
    first = upcoming.sort("kickoff").row(0, named=True)
    return upcoming.filter((pl.col("season") == first["season"]) & (pl.col("week") == first["week"]))


def active_players(week_games):
    """Who's expected to play: the listed starting QB, plus running backs, receivers and tight ends on the active roster,
    minus anyone listed Out or Doubtful on the injury report or named in PLAYERS_OUT."""
    season, week = week_games["season"][0], week_games["week"][0]
    rosters = nfl.load_rosters_weekly([season]).with_columns(pl.col("team").replace(RELOCATED))
    roster_week = rosters.filter(pl.col("week") <= week)["week"].max()
    roster = rosters.filter((pl.col("week") == roster_week) & (pl.col("status") == "ACT")
                            & pl.col("position").is_in(["RB", "FB", "WR", "TE"]))
    returning = set(rosters.filter(pl.col("status") == "RES")["gsis_id"].drop_nulls())   # back from a reserve list
    try:
        report = nfl.load_injuries([season]).filter((pl.col("week") == week) & pl.col("report_status").is_in(["Out", "Doubtful"]))
        injured = set(report["gsis_id"].drop_nulls())
    except Exception:
        injured = set()
    played_this_season = (
        pl.concat([games.select("season", "result", team="home_team"), games.select("season", "result", team="away_team")])
        .filter((pl.col("season") == season) & pl.col("result").is_not_null())
        .group_by("team").len()
    )
    team_games = dict(played_this_season.iter_rows())
    out_names = set(PLAYERS_OUT)

    rows = []
    for g in week_games.iter_rows(named=True):
        home_qb, away_qb = starter_by_game.get(g["game_id"], (None, None))
        for team, starter in ((g["home_team"], home_qb), (g["away_team"], away_qb)):
            ids = set(roster.filter(pl.col("team") == team)["gsis_id"].drop_nulls())
            keep = [p for p in ids if p in player_totals and (
                player_last_season.get(p) == season or team_games.get(team, 0) < 2 or p in returning)]
            if starter in player_totals:
                keep.append(starter)
            for p in set(keep):
                if p in injured or player_names.get(p) in out_names:
                    continue
                snap = player_snapshot(player_totals[p], player_positions.get(p), season, player_last_season.get(p), **PROP_PLAYER_SETTINGS)
                rows.append({"game_id": g["game_id"], "team": team, "player_id": p, "player": player_names.get(p),
                             "position": "QB" if p == starter else player_positions.get(p),
                             "games_before": player_games.get(p, 0), **snap})
    return pl.DataFrame(rows, infer_schema_length=None)


now = datetime.now(EASTERN)
prop_week = next_week_games(now)
if prop_week.is_empty():
    week_props = pl.DataFrame()
    print("No upcoming games found.")
else:
    lines = pl.DataFrame(
        [{"player": k[0], "stat": k[1], "line": float(v)} for k, v in PROP_LINES.items()],
        schema={"player": pl.String, "stat": pl.String, "line": pl.Float64},
    )
    book = book_lines(prop_week, now)         # sportsbook lines from The Odds API, from the cell above
    week_props = (
        project_props(prop_table(active_players(prop_week), live_team_proj), live_prop_models)
        .join(prop_week.select("game_id", "week", "kickoff"), on="game_id")
        .join(lines, on=["player", "stat"], how="left")
        .with_columns(name_key=pl.col("player").map_elements(name_key, return_dtype=pl.String))
        .join(book, on=["game_id", "name_key", "stat"], how="left")
        .with_columns(line_source=pl.when(pl.col("line").is_not_null()).then(pl.lit("you"))
                      .when(pl.col("book_line").is_not_null()).then(pl.lit("sportsbooks")))
        .with_columns(
            line=pl.coalesce("line", "book_line"),                  # a line you typed in wins over the sportsbooks'
            over_price=pl.when(pl.col("line_source") == "sportsbooks").then(pl.col("over_price")),
            under_price=pl.when(pl.col("line_source") == "sportsbooks").then(pl.col("under_price")),
            books=pl.when(pl.col("line_source") == "sportsbooks").then(pl.col("books")),
        )
        .drop("name_key", "book_line")
    )
    week_props = add_over_chance(week_props, live_prop_models)
    unmatched = lines.join(week_props, on=["player", "stat"], how="anti")
    for r in unmatched.iter_rows(named=True):
        print(f"No projection for {r['player']} ({r['stat']}). Check the spelling, or he may not have a prop this week.")

    show = week_props if not SHOW_TEAMS else week_props.filter(pl.col("team").is_in(SHOW_TEAMS) | pl.col("opp").is_in(SHOW_TEAMS))
    print(f"{prop_week['season'][0]} season, week {prop_week['week'][0]}: {week_props.height} props for {week_props['player_id'].n_unique()} players.")
    print("Projection = average outcome. 50/50 line = half the time he finishes above it. Likely range = the middle half of outcomes.")
    print("Line = the sportsbooks' line (or yours), over_price = the middle of their prices for the over, over_chance = the model's chance of going over.\n")
    for stat, name in PROP_NAMES.items():
        t = show.filter(pl.col("stat") == stat).sort("projection", descending=True)
        print(f"{name}:")
        with pl.Config(tbl_rows=-1):
            display(t.select(
                "player", "team", "opp",
                projection=pl.col("projection").round(1),
                line_50_50=pl.col("median").round(1),
                likely_range=pl.format("{}–{}", pl.col("low").round(0).cast(pl.Int64), pl.col("high").round(0).cast(pl.Int64)),
                line=pl.col("line"),
                over_price=pl.col("over_price"),
                over_chance=pl.col("p_over").round(3),
            ))

## Step 11: Save this week's props

Props go in their own log, `nfl_props_log.csv`, next to your pick log in Google Drive.

- **Props update until kickoff.** Each run saves the props for games that haven't kicked off, replacing that game's earlier props, so they pick up the latest injury news and any lines you've added. Once a game starts, its props are locked. (Game picks in Step 8 work differently: the first logged pick counts.)
- **After the games**, it fills in what each player did and grades the model against his recent average. For props with a sportsbook line, from The Odds API or typed in, it also keeps a win-loss record against that line.
- **Inactive players don't count.** A player with no stats in a finished game is left out of the grading, the way a sportsbook voids a bet on a player who doesn't play.

In [ ]:
PROPS_MODEL = "props_v1"
PROPS_LOG_PATH = LOG_PATH.parent / "nfl_props_log.csv"
PROPS_COLUMNS = {
    "logged_at_et": pl.String, "model": pl.String, "season": pl.Int32, "week": pl.Int32,
    "game_id": pl.String, "kickoff_et": pl.String, "player_id": pl.String, "player": pl.String,
    "team": pl.String, "opp": pl.String, "stat": pl.String, "recent_avg": pl.Float64,
    "projection": pl.Float64, "median": pl.Float64, "low": pl.Float64, "high": pl.Float64,
    "line": pl.Float64, "p_over": pl.Float64,
}


def read_props_log():
    if not PROPS_LOG_PATH.exists():
        return pl.DataFrame(schema=PROPS_COLUMNS)
    log = pl.read_csv(PROPS_LOG_PATH, infer_schema_length=0)        # read as text, then fix the types
    return log.select([pl.col(c).cast(t) for c, t in PROPS_COLUMNS.items()])


def update_props_log(week_props, now=None):
    """Save this run's props for games that haven't kicked off, fill in results, and save the log."""
    now = now or datetime.now(EASTERN)
    log = read_props_log()
    added = pl.DataFrame(schema=PROPS_COLUMNS)
    if not week_props.is_empty():
        added = week_props.filter(pl.col("kickoff") > now).select(
            logged_at_et=pl.lit(now.astimezone(EASTERN).strftime("%Y-%m-%d %H:%M")),
            model=pl.lit(PROPS_MODEL),
            season=pl.col("season"), week=pl.col("week"), game_id=pl.col("game_id"),
            kickoff_et=pl.col("kickoff").dt.strftime("%Y-%m-%d %H:%M"),
            player_id=pl.col("player_id"), player=pl.col("player"), team=pl.col("team"), opp=pl.col("opp"),
            stat=pl.col("stat"),
            **{c: pl.col(c).round(1) for c in ("recent_avg", "projection", "median", "low", "high")},
            line=pl.col("line"), p_over=pl.col("p_over").round(3),
        ).cast(PROPS_COLUMNS)
    # Games that haven't kicked off get this run's projections; games that have started stay locked
    log = pl.concat([log.filter(~pl.col("game_id").is_in(added["game_id"].unique().to_list())), added])
    results = box.select("game_id", "player_id", *[pl.col(s).cast(pl.Float64) for s in PROP_STATS]).unpivot(
        index=["game_id", "player_id"], variable_name="stat", value_name="actual")
    log = log.join(results, on=["game_id", "player_id", "stat"], how="left").sort("kickoff_et", "game_id", "stat", "player")
    log.write_csv(PROPS_LOG_PATH)
    return log, added


def props_scorecard(log):
    """Grade logged props that have results. Players with no stats in a finished game are left out, like a void bet."""
    graded = log.filter(pl.col("actual").is_not_null())
    if graded.is_empty():
        print("No logged props have results yet. Run this again after the week's games to grade them.")
        return
    print(f"Graded props from {graded['kickoff_et'].min()[:10]} to {graded['kickoff_et'].max()[:10]}.")
    display(graded.group_by("stat").agg(
        pl.len().alias("graded_props"),
        (pl.col("projection") - pl.col("actual")).abs().mean().round(2).alias("model_miss"),
        (pl.col("recent_avg") - pl.col("actual")).abs().mean().round(2).alias("recent_average_miss"),
    ).with_columns(prop=pl.col("stat").replace(PROP_NAMES)).sort("prop").select("prop", "graded_props", "model_miss", "recent_average_miss"))

    bets = graded.filter(pl.col("line").is_not_null() & pl.col("p_over").is_not_null() & (pl.col("p_over") != 0.5)).with_columns(
        pick_over=pl.col("p_over") > 0.5,
        vs_line=pl.col("actual") - pl.col("line"),
    )
    if bets.is_empty():
        print("No graded props with a sportsbook line yet. Lines come from The Odds API (Step 10) or from PROP_LINES.")
        return
    print("Props with a sportsbook line (the model takes whichever side it gives better than 50%). Break-even at -110 is 52.4%:")
    display(bets.group_by("stat").agg(
        ((pl.col("vs_line") != 0) & ((pl.col("vs_line") > 0) == pl.col("pick_over"))).sum().alias("wins"),
        ((pl.col("vs_line") != 0) & ((pl.col("vs_line") > 0) != pl.col("pick_over"))).sum().alias("losses"),
        (pl.col("vs_line") == 0).sum().alias("pushes"),
    ).with_columns(
        prop=pl.col("stat").replace(PROP_NAMES),
        win_rate=(pl.col("wins") / (pl.col("wins") + pl.col("losses"))).round(3),
    ).select("prop", "wins", "losses", "pushes", "win_rate"))


props_log, props_added = update_props_log(week_props)
print(f"Props log: {PROPS_LOG_PATH} ({props_log.height} props logged so far)")
if props_added.is_empty():
    print("Nothing new to log: this week's games have all kicked off, or none are scheduled.\n")
else:
    print(f"Saved {props_added.height} props for {props_added['game_id'].n_unique()} games that haven't kicked off, "
          f"{props_added['season'][0]} season, week {props_added['week'][0]}.\n")
props_scorecard(props_log)

## Step 12: Save the website data

The picks website reads one file per league. This step writes `app_data/nfl.json` in your Google Drive, next to your pick log. It holds:
- **This week's games**: the model's numbers, the sportsbooks' numbers, the starting quarterbacks, and the final score for games that are over.
- **Player props** for the games that haven't kicked off, with the sportsbooks' line and prices when there are any.
- **What the Top picks lists need**: each game's chance to cover the spread, found the same way Step 6 finds the over chance, and each player's chance of reaching common milestones, like 250+ passing yards or 100+ receiving yards. The repository's `scripts/top_picks.py` builds the website's Top picks lists from them.
- **How the models are doing**: the season record and every graded game this season. They're graded from the pick log, the same way Step 8 grades them. When `game_review_v1` has reviewed a game, its explanation comes along too.

- **The report card**: how accurate the picks have been on the model's own terms: winners picked, the average miss on the margin and the total, and whether its win chances came true. Next to each is the same number from the 2020–2025 test seasons in Step 5, so you can see whether this season is better or worse than usual. The sportsbooks' numbers are included only for reference.

Spreads in the file are written from the home team's side: a `home_margin` of 2.5 means the home team is expected to win by 2.5. The website turns that into sportsbook style, like *JAX -2.5*. Kickoff times include their time zone, so the website shows them in each visitor's own time.

**The website itself** is `index.html` in the same `app_data` folder. Once it's online, it reads `nfl.json` and `cfb.json` from its own folder. To look at it before then, download `index.html`, open it in your browser and drop the two `.json` files onto the page.

In [ ]:
import json

APP_FOLDER = LOG_PATH.parent / "app_data"           # the website's data, next to your pick log
APP_FILE = APP_FOLDER / "nfl.json"
REVIEW_FILE = LOG_PATH.parent / "nfl_game_reviews.csv"   # written by game_review_v1
ELO_PREFIX = "elo_v"                                 # Elo is logged but isn't the main model, as in the review


def fit_cover_chance(scored, seasons):
    """How often the home team covered, given how far the model's spread was from the betting line. Built like the over chance in Step 6."""
    t = scored.filter(
        pl.col("season").is_between(*seasons) & pl.col("result").is_not_null()
        & pl.col("spread_line").is_not_null() & (pl.col("result") != pl.col("spread_line"))
    )
    gap = (t["model_spread"] - t["spread_line"]).to_numpy().reshape(-1, 1)
    return LogisticRegression(C=1e6).fit(gap, (t["result"] > t["spread_line"]).to_numpy().astype(int))


cover_model = fit_cover_chance(live, LIVE_SEASONS)


def cover_chance(g):
    """The model's chance that the home team covers the sportsbook spread. The Top picks lists use it."""
    if g["spread_line"] is None or g["model_spread"] is None:
        return None
    return float(cover_model.predict_proba([[g["model_spread"] - g["spread_line"]]])[0, 1])


# "At least" lines for each prop, like a sportsbook's alternate lines. The Top picks lists use them.
MILESTONES = {
    "passing_yards": [200, 225, 250, 275, 300, 325, 350],
    "rushing_yards": [40, 50, 60, 70, 80, 90, 100, 125, 150],
    "receiving_yards": [40, 50, 60, 70, 80, 90, 100, 125, 150],
    "receptions": [3, 4, 5, 6, 7, 8, 9, 10],
}


def milestone_chances(r):
    """His chance of reaching each milestone: the share of similar past players who did, scaled to his projection, as in Step 9."""
    proj = r["projection"]
    if proj is None or proj <= 0:
        return None
    ratios = live_prop_models[r["stat"]]["ratios"][r["group"]]
    return {str(t): round(1 - np.searchsorted(ratios, t / proj, side="left") / len(ratios), 3) for t in MILESTONES[r["stat"]]}


def clean(value, digits=None):
    """A value ready for the website: numbers rounded, missing values as null."""
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return None
    if isinstance(value, float) and digits is not None:
        return round(value, digits)
    return value


def side_result(model_line, market_line, actual):
    """Win, loss or push for the side the model liked compared with the sportsbook line."""
    if model_line is None or market_line is None or actual is None:
        return None
    if model_line == market_line:
        return "no pick"
    if actual == market_line:
        return "push"
    return "win" if (model_line > market_line) == (actual > market_line) else "loss"


def logged_picks(log):
    """One row per logged game: the newest margin pick (Elo aside), the totals pick and the final score."""
    margin = (log.filter(pl.col("model_spread").is_not_null() & ~pl.col("model").str.starts_with(ELO_PREFIX))
              .sort("logged_at_et").group_by("game_id").agg(
                  margin_model=pl.col("model").last(), pick_margin=pl.col("model_spread").last(),
                  pick_win=pl.col("model_p_home").last(), line_margin=pl.col("market_spread").last()))
    totals = (log.filter(pl.col("model_total").is_not_null()).sort("logged_at_et").group_by("game_id").agg(
                  pick_total=pl.col("model_total").last(), line_total=pl.col("market_total").last()))
    games = log.sort("logged_at_et").group_by("game_id").agg(
        pl.col("season").last(), pl.col("week").last(), pl.col("kickoff_et").last(),
        pl.col("away_team").last(), pl.col("home_team").last(),
        away_final=pl.col("away_score").last(), home_final=pl.col("home_score").last())
    return games.join(margin, on="game_id", how="left").join(totals, on="game_id", how="left")


def graded_results(picks, season, reviews):
    """Every finished game this season that the models picked, graded the same way as the pick log."""
    out = []
    done = picks.filter((pl.col("season") == season) & pl.col("home_final").is_not_null()).sort("kickoff_et", "game_id", descending=True)
    for r in done.iter_rows(named=True):
        margin, total = r["home_final"] - r["away_final"], r["home_final"] + r["away_final"]
        pick = r["pick_margin"]
        review = reviews.get(str(r["game_id"]), {})
        out.append({
            "id": r["game_id"], "week": r["week"],
            "kickoff": datetime.strptime(r["kickoff_et"], "%Y-%m-%d %H:%M").replace(tzinfo=EASTERN).isoformat(),
            "away": r["away_team"], "home": r["home_team"],
            "final": {"away": r["away_final"], "home": r["home_final"]},
            "model": {"home_margin": clean(pick, 1), "home_win": clean(r["pick_win"], 3), "total": clean(r["pick_total"], 1)},
            "vegas": {"home_margin": clean(r["line_margin"]), "total": clean(r["line_total"])},
            "winner": None if pick is None or pick == 0 or margin == 0 else (pick > 0) == (margin > 0),
            "ats": side_result(pick, r["line_margin"], margin),
            "over_under": side_result(r["pick_total"], r["line_total"], total),
            "review": review.get("summary") or None,
            "total_review": review.get("total_summary") or None,
        })
    return out


def tally(results):
    """Season-style record for a list of graded games."""
    def count(key):
        values = [r[key] for r in results]
        return {"win": values.count("win"), "loss": values.count("loss"), "push": values.count("push")}

    picked = [r for r in results if r["winner"] is not None]
    spread_games = [r for r in results if r["model"]["home_margin"] is not None and r["vegas"]["home_margin"] is not None]
    total_games = [r for r in results if r["model"]["total"] is not None and r["vegas"]["total"] is not None]

    def miss(games, key, side):
        if not games:
            return None
        actual = (lambda g: g["final"]["home"] - g["final"]["away"]) if key == "home_margin" else (lambda g: g["final"]["home"] + g["final"]["away"])
        return round(sum(abs(actual(g) - g[side][key]) for g in games) / len(games), 1)

    return {
        "games": len(results),
        "winners": {"right": sum(r["winner"] for r in picked), "picked": len(picked)},
        "ats": count("ats"), "over_under": count("over_under"),
        "spread_miss": {"model": miss(spread_games, "home_margin", "model"), "vegas": miss(spread_games, "home_margin", "vegas"), "games": len(spread_games)},
        "total_miss": {"model": miss(total_games, "total", "model"), "vegas": miss(total_games, "total", "vegas"), "games": len(total_games)},
    }


def game_status(g, now):
    if g["result"] is not None:
        return "final"
    return "upcoming" if g["kickoff"] > now else "in_progress"


now = datetime.now(EASTERN)
timed = kickoff_times(live)
ahead = timed.filter(pl.col("result").is_null() & (pl.col("kickoff") > now))
anchor = ahead.sort("kickoff").row(0, named=True) if ahead.height else timed.filter(pl.col("result").is_not_null()).sort("kickoff").row(-1, named=True)
this_week = timed.filter((pl.col("season") == anchor["season"]) & (pl.col("week") == anchor["week"])).sort("kickoff", "game_id")

try:
    TEAM_NAMES = dict(nfl.load_teams().select("team_abbr", "team_name").iter_rows())
except Exception:
    TEAM_NAMES = {}

games_out = []
for g in this_week.iter_rows(named=True):
    status = game_status(g, now)
    games_out.append({
        "id": g["game_id"], "kickoff": g["kickoff"].isoformat(), "status": status,
        "away": g["away_team"], "home": g["home_team"],
        "away_name": TEAM_NAMES.get(g["away_team"], g["away_team"]), "home_name": TEAM_NAMES.get(g["home_team"], g["home_team"]),
        "neutral": g["location"] != "Home",
        "model": {"home_margin": clean(g["model_spread"], 1), "home_win": clean(g["model_p_home"], 3),
                  "total": clean(g["model_total"], 1), "over": clean(g["model_p_over"], 3),
                  "home_cover": clean(cover_chance(g), 3)},
        "vegas": {"home_margin": clean(g["spread_line"]), "home_win": clean(g["mkt_p_home"], 3),
                  "total": clean(g["total_line"]), "over": clean(g["mkt_p_over"], 3),
                  "home_moneyline": g["home_moneyline"], "away_moneyline": g["away_moneyline"]},
        "quarterbacks": {"away": qb_name(g["away_qb"]) or None, "home": qb_name(g["home_qb"]) or None},
        "final": {"away": g["away_score"], "home": g["home_score"]} if status == "final" else None,
    })

props_out = []
if not week_props.is_empty():
    for r in week_props.sort("stat", "projection", descending=[False, True]).iter_rows(named=True):
        props_out.append({
            "game": r["game_id"], "player_id": r["player_id"], "player": r["player"], "position": r["position"], "team": r["team"], "opp": r["opp"],
            "stat": r["stat"], "projection": clean(r["projection"], 1), "line_50_50": clean(r["median"], 1),
            "low": clean(r["low"], 0), "high": clean(r["high"], 0),
            "line": clean(r["line"]), "line_source": r["line_source"], "over_price": r["over_price"], "under_price": r["under_price"],
            "books": r["books"], "over_chance": clean(r["p_over"], 3),
            "milestones": milestone_chances(r),
        })

reviews = {}
if REVIEW_FILE.exists():
    reviews = {str(r["game_id"]): r for r in pl.read_csv(REVIEW_FILE, infer_schema_length=0).iter_rows(named=True)}
results = graded_results(logged_picks(log), anchor["season"], reviews)
weeks = sorted({r["week"] for r in results}, reverse=True)

# The report card: how accurate the picks were on the model's own terms, next to the out-of-sample test seasons
# from Step 5. The sportsbooks' numbers ride along only for reference.
CONFIDENCE = [(0.6, "50–60%"), (0.7, "60–70%"), (0.8, "70–80%"), (0.9, "80–90%"), (1.01, "90%+")]


def accuracy(rows):
    """Winners picked, how far off the margin and total were, and the Brier score of the win chances (0 is perfect,
    0.25 is a coin flip). Each row has the model's home win chance p, margin m and total t, the sportsbooks' margin
    vm and total vt, and the final margin and total."""
    def avg(values, digits):
        values = [v for v in values if v is not None]
        return round(sum(values) / len(values), digits) if values else None
    decided = [r for r in rows if r["p"] is not None and r["margin"] != 0]
    return {
        "games": len(rows),
        "winners": avg([float((r["p"] >= 0.5) == (r["margin"] > 0)) for r in decided], 3),
        "brier": avg([(r["p"] - float(r["margin"] > 0)) ** 2 for r in decided], 3),
        "margin_miss": avg([abs(r["m"] - r["margin"]) for r in rows if r["m"] is not None], 1),
        "total_miss": avg([abs(r["t"] - r["total"]) for r in rows if r["t"] is not None], 1),
        "vegas_margin_miss": avg([abs(r["vm"] - r["margin"]) for r in rows if r["vm"] is not None and r["m"] is not None], 1),
        "vegas_total_miss": avg([abs(r["vt"] - r["total"]) for r in rows if r["vt"] is not None and r["t"] is not None], 1),
    }


def calibration(rows):
    """For each level of confidence: how many games, the chance the model gave its pick on average, and how often it won."""
    groups = {label: [] for _, label in CONFIDENCE}
    for r in rows:
        if r["p"] is None or r["margin"] == 0:
            continue
        fav = max(r["p"], 1 - r["p"])
        label = next(name for top, name in CONFIDENCE if fav <= top)
        groups[label].append((fav, float((r["p"] >= 0.5) == (r["margin"] > 0))))
    return [{"label": label, "games": len(g),
             "predicted": round(sum(f for f, _ in g) / len(g), 3) if g else None,
             "actual": round(sum(w for _, w in g) / len(g), 3) if g else None} for label, g in groups.items()]


bench = (
    scored.filter(pl.col("season").is_between(*TEST) & pl.col("result").is_not_null() & pl.col("spread_line").is_not_null())
    .select("game_id", "model_p_home", "model_spread", "spread_line", "result")
    .join(ttest.select("game_id", "model_total", "total_line", "total"), on="game_id", how="left")
)
bench_rows = [{"p": r["model_p_home"], "m": r["model_spread"], "t": r["model_total"], "vm": r["spread_line"], "vt": r["total_line"],
               "margin": r["result"], "total": r["total"]} for r in bench.iter_rows(named=True)]
live_rows = [{"week": r["week"], "p": r["model"]["home_win"], "m": r["model"]["home_margin"], "t": r["model"]["total"],
              "vm": r["vegas"]["home_margin"], "vt": r["vegas"]["total"],
              "margin": r["final"]["home"] - r["final"]["away"], "total": r["final"]["home"] + r["final"]["away"]} for r in results]
report = {
    "benchmark": {"seasons": f"{TEST[0]}–{TEST[1]}", **accuracy(bench_rows), "calibration": calibration(bench_rows)},
    "season": {**accuracy(live_rows), "calibration": calibration(live_rows)},
    "weeks": [{"week": w, **accuracy([r for r in live_rows if r["week"] == w])} for w in sorted({r["week"] for r in live_rows})],
}

app_data = {
    "league": "NFL",
    "updated": now.isoformat(timespec="minutes"),
    "season": anchor["season"], "week": anchor["week"], "week_label": f"Week {anchor['week']}",
    "models": {"margin": "elo_qb_rest_v1", "totals": "totals_v1", "props": PROPS_MODEL, "cover": "cover_chance_v1"},
    "games": games_out,
    "props": props_out,
    "record": {"season": tally(results), "weeks": [{"week": w, **tally([r for r in results if r["week"] == w])} for w in weeks]},
    "results": results,
    "report": report,
}
APP_FOLDER.mkdir(parents=True, exist_ok=True)
APP_FILE.write_text(json.dumps(app_data, indent=1))

reviewed = sum(r["review"] is not None for r in results)
print(f"Saved the website data to {APP_FILE}:")
print(f"  {len(games_out)} games for {anchor['season']} week {anchor['week']}, with cover chances, and {len(props_out)} player props with milestone chances")
print(f"  {len(results)} graded games this season" + (f", {reviewed} with a post-game review" if results else ""))
b, s = report["benchmark"], report["season"]
print(f"  Report card: average margin miss {s['margin_miss']} this season vs. {b['margin_miss']} in the {b['seasons']} tests; "
      f"winners {s['winners']} vs. {b['winners']}" if s["games"] else f"  Report card: no graded games yet ({b['seasons']} tests: average margin miss {b['margin_miss']})")
if results and not REVIEW_FILE.exists():
    print(f"  No review file yet at {REVIEW_FILE}. Run game_review_v1 before this notebook to add the post-game explanations.")

## Next steps

The weekly runs, the website, the Top picks lists and the AI news check now happen in the `football-picks` repository. Its README explains them.

1. **Test the props model against sportsbook lines.** Once a few weeks of lines are in the props log, check whether the model's over chances beat the 52.4% break-even, the same way Steps 5 and 6 test the game models.
2. **More props.** Touchdowns, completions and attempts, using the same team-then-player approach.